# 🌊 Ocean Dataset Downloader — Google Colab

Fully self-contained — the `ocean_downloader` package source is embedded directly in this notebook (no zip upload or GitHub clone required).

Downloads four oceanographic datasets:

| Dataset | Output folder |
|---|---|
| CMEMS Altimetry (SLA/ADT) | `CMEMS_ALTIMETRY/` |
| CMEMS Sea Surface Salinity | `CMEMS_SSS/` |
| ERA5 10-m Winds | `ERA5_WINDS/` |
| NOAA OISST AVHRR SST | `NOAA_OISST_AVHRR/` |

**Steps:**
1. Run **Setup** cells (install deps, setup storage, load package)
2. Fill in **Credentials**
3. Set your **Date Range**
4. Run whichever dataset cells you need

---
## 1 — Setup

In [1]:
# ── Install dependencies ──────────────────────────────────────────────────────
# copernicusmarine : CMEMS Altimetry + SSS
# cdsapi           : ERA5 winds
# xarray / netCDF4 : reading & writing NetCDF files
# numpy            : ERA5 wind-speed derivation
%pip install -q copernicusmarine cdsapi xarray netCDF4 numpy
import tensorflow as tf
print('✅ Dependencies installed.')

✅ Dependencies installed.


In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  tf.config.experimental.set_memory_growth(gpus[0], True)
  print(gpus[0])

PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [3]:
# ── Setup Local Storage ───────────────────────────────────────────────────────
# All downloaded data will live under OUTPUT_ROOT defined below.
# ⚠️ NOTE: If in Colab, files in /content/ are temporary. Download them!

import sys
from pathlib import Path
if "google.colab" in sys.modules:
    OUTPUT_ROOT = Path("/content/ocean_data")
else:
    OUTPUT_ROOT = Path("./ocean_data").resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"📂 Output root: {OUTPUT_ROOT}")
import os
os.environ["OCEAN_DATA_ROOT"] = str(OUTPUT_ROOT)


📂 Output root: /content/ocean_data


---
## 2 — Credentials

**Preferred (secure):** add your secrets in *Runtime → Secrets* (🔑 icon in the left sidebar):
- `CMEMS_USER` and `CMEMS_PASS` — register free at https://marine.copernicus.eu/
- `CDS_API_KEY` — get yours at https://cds.climate.copernicus.eu/

If secrets are not found, the cell falls back to interactive password prompts.

In [4]:
import os
from getpass import getpass

def _colab_secret(name):
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None

if not os.environ.get("CMEMS_USER"):
    os.environ["CMEMS_USER"] = (
        _colab_secret("CMEMS_USER")
        or input("CMEMS username (register free at https://marine.copernicus.eu/): ")
    )
if not os.environ.get("CMEMS_PASS"):
    os.environ["CMEMS_PASS"] = _colab_secret("CMEMS_PASS") or getpass("CMEMS password: ")
if not os.environ.get("CDS_API_KEY"):
    os.environ["CDS_API_KEY"] = _colab_secret("CDS_API_KEY") or getpass("CDS API key: ")

print("✅ Credentials set in environment")


def load_cmems_credentials():
    user = os.environ.get("CMEMS_USER")
    pwd  = os.environ.get("CMEMS_PASS")
    if not user or not pwd:
        raise RuntimeError("Set CMEMS_USER and CMEMS_PASS (Colab Secrets or env).")
    return user, pwd


CMEMS username (register free at https://marine.copernicus.eu/): namanjain0313@gmail.com
CMEMS password: ··········
CDS API key: ··········
✅ Credentials set in environment


---
## 4 — Download Datasets

Run **any combination** of the four cells below. Each is independent.

In [5]:
from pathlib import Path
from datetime import date
import calendar

REGION = dict(min_lat=5.0, max_lat=30.0, min_lon=45.0, max_lon=105.0)

def _days_in_month(year, month):
    return calendar.monthrange(year, month)[1]

In [6]:
import xarray as xr
import copernicusmarine

ALT_MY_DATASET = "cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D"
ALT_NRT_DATASET = "cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D"
ALT_VARS = ["sla", "adt", "ugos", "vgos", "ugosa", "vgosa"]
ALT_NRT_START = (2024, 1)

def _alt_candidates(year, month):
    if (year, month) < ALT_NRT_START:
        return [ALT_MY_DATASET, ALT_NRT_DATASET]
    return [ALT_NRT_DATASET, ALT_MY_DATASET]

def _alt_regrid_025(ds):
    """Coarsen native 0.125° → 0.25° (2×2 mean)."""
    ds_025 = ds.coarsen(latitude=2, longitude=2, boundary="exact").mean(keep_attrs=True)
    # Print resolution check
    lat_res = float(ds_025.latitude[1] - ds_025.latitude[0])
    lon_res = float(ds_025.longitude[1] - ds_025.longitude[0])
    print(f"  Regridded grid: {ds_025.sizes['latitude']} × {ds_025.sizes['longitude']} "
          f"@ {lat_res:.4f}° × {lon_res:.4f}°")
    return ds_025

def download_altimetry_month(year, month, username, password,
                             outdir=OUTPUT_ROOT / "CMEMS_ALTIMETRY"):
    n_days = _days_in_month(year, month)
    tmp = outdir / f"_tmp_alt_{year}{month:02d}.nc"
    tmp.parent.mkdir(parents=True, exist_ok=True)
    if tmp.exists():
        tmp.unlink()

    dataset_used, last_err = None, None
    for dataset_id in _alt_candidates(year, month):
        try:
            copernicusmarine.subset(
                dataset_id=dataset_id,
                variables=ALT_VARS,
                start_datetime=f"{year}-{month:02d}-01T00:00:00",
                end_datetime=f"{year}-{month:02d}-{n_days:02d}T23:59:59",
                minimum_latitude=REGION["min_lat"],
                maximum_latitude=REGION["max_lat"],
                minimum_longitude=REGION["min_lon"],
                maximum_longitude=REGION["max_lon"],
                output_filename=tmp.name, output_directory=str(tmp.parent),
                username=username, password=password,
                overwrite=True,
            )
            dataset_used = dataset_id
            break
        except Exception as e:
            last_err = e
            print(f"  ⚠️ {dataset_id} failed: {type(e).__name__}: {e}")
            if tmp.exists():
                tmp.unlink()
    if dataset_used is None:
        print(f"❌ Altimetry {year}-{month:02d} failed: {last_err}")
        return

    n_saved = 0
    try:
        ds = xr.open_dataset(tmp)
        for day in range(1, n_days + 1):
            date_str = f"{year}-{month:02d}-{day:02d}"
            outfile = (outdir / f"{year}" / f"{month:02d}"
                       / f"altimetry_{year}{month:02d}{day:02d}.nc")
            if outfile.exists():
                continue
            outfile.parent.mkdir(parents=True, exist_ok=True)
            try:
                daily = ds.sel(time=date_str, method="nearest")
                daily = _alt_regrid_025(daily)   # <-- regrid to 0.25°
                daily.to_netcdf(outfile)
                n_saved += 1
            except Exception as e:
                print(f"  skip {date_str}: {e}")
        ds.close()
    finally:
        if tmp.exists():
            tmp.unlink()
    print(f"✅ Altimetry {year}-{month:02d} done "
          f"({n_saved} new, {dataset_used}) → {outdir}")


In [7]:
# ── 4b. CMEMS Sea Surface Salinity ─────────────────────────────────────────────
#
# Product (best for Indian Ocean + 0.25° + stretches to present):
#   MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013
#   Multi Observation Global Ocean SSS/SSD, CNR — daily gap-free L4 (SMOS+SMAP +
#   in-situ + SST via multivariate OI). Global, incl. 5°N–30°N, 45°E–105°E.
#
# Datasets stitched (same processor, seamless overlap 2024-01-01 → 2024-12-15):
#   cmems_obs-mob_glo_phy-sss_my_multi_P1D   1993-01-01 → 2024-12-15 (use iff
#                                            requested month ends < 2024-01-01)
#   cmems_obs-mob_glo_phy-sss_nrt_multi_P1D  2024-01-01 → present, ~6-day latency
#                                            (use for any month touching >= 2024-01-01)
#
# Native grid: 0.125° × 0.125° daily → coarsened 2×2 mean to 0.25° × 0.25° here
# to match the altimetry / ERA5 grid.
# Native variable is `sos` (sea_surface_salinity); saved as `sss` for stable
# downstream naming. Output: CMEMS_SSS/YYYY/MM/sss_YYYYMMDD.nc (one file/day).
#
# FIXES vs previous version:
#   - `copernicusmarine.get(product_id=...)` → removed (get() takes dataset_id
#     only; catalogue lookup is `describe()`). Dataset IDs are now hardcoded
#     after catalogue verification instead of guessed at runtime.
#   - placeholder "cmems_obs-mob_glo_phy-sss_mynrt_smos-sm...?" removed.
#   - variables=["sss"] → ["sos"] (actual short_name), with auto-detect fallback.
#   - coords latitude/longitude (+depth) normalized; numpy-datetime64 timestamps
#     handled via pandas (old `hasattr(ts,'strftime')` dropped valid times).
#   - MY-only weekly product (ends Month-5) replaced by MY+NRT daily stitch.

import copernicusmarine
import xarray as xr

SSS_PRODUCT_ID = "MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013"
SSS_MY_DATASET = "cmems_obs-mob_glo_phy-sss_my_multi_P1D"
SSS_NRT_DATASET = "cmems_obs-mob_glo_phy-sss_nrt_multi_P1D"
SSS_NRT_START = (2024, 1)  # (year, month): months >= this use NRT
SSS_NATIVE_VAR = "sos"  # verified via describe(); fallback auto-detects sss/sos
SSS_VAR_CANDIDATES = ("sos", "sss")


def _sss_candidate_datasets(year, month):
    """Primary-first dataset routing so history→present is seamless."""
    if (year, month) < SSS_NRT_START:
        return [SSS_MY_DATASET, SSS_NRT_DATASET]
    return [SSS_NRT_DATASET, SSS_MY_DATASET]


def _sss_download_subset(dataset_id, year, month, n_days, username, password, tmp):
    """Download one monthly spatial subset; retry without var filter if CMEMS
    rejects the variable name (future-proofing). Returns True on success."""
    common = dict(
        dataset_id=dataset_id,
        start_datetime=f"{year}-{month:02d}-01T00:00:00",
        end_datetime=f"{year}-{month:02d}-{n_days:02d}T23:59:59",
        minimum_latitude=REGION["min_lat"], maximum_latitude=REGION["max_lat"],
        minimum_longitude=REGION["min_lon"], maximum_longitude=REGION["max_lon"],
        output_filename=tmp.name, output_directory=str(tmp.parent),
        username=username, password=password,
        overwrite=True,
    )
    try:
        copernicusmarine.subset(variables=[SSS_NATIVE_VAR], **common)
        return True
    except Exception as e:
        msg = str(e).lower()
        if "variable" not in msg:
            raise
        print(f"  ⚠️ variable '{SSS_NATIVE_VAR}' rejected, retrying without filter: {e}")
        copernicusmarine.subset(**common)
        return True


def _sss_normalize(ds):
    """Normalize coords/vars: latitude→lat, longitude→lon, drop depth, pick SSS var."""
    rename = {}
    if "latitude" in ds.coords:
        rename["latitude"] = "lat"
    if "longitude" in ds.coords:
        rename["longitude"] = "lon"
    if rename:
        ds = ds.rename(rename)
    # Depth is singleton (0 m) on this product — drop it for 2D daily files.
    if "depth" in ds.coords:
        try:
            ds = ds.sel(depth=0, drop=True)
        except Exception:
            ds = ds.isel(depth=0, drop=True) if "depth" in ds.dims else ds.drop_vars("depth", errors="ignore")
    elif "depth" in ds.dims:
        ds = ds.isel(depth=0, drop=True)
    sss_var = next((v for v in SSS_VAR_CANDIDATES if v in ds.data_vars), None)
    if sss_var is None:
        raise KeyError(f"SSS variable not found. Available: {list(ds.data_vars)}")
    return ds, sss_var


def _sss_regrid_025(ds):
    """Coarsen native 0.125° → 0.25° (2×2 mean); fallback to interp on exact grid."""
    import numpy as np
    lat_res = abs(float(ds["lat"][1] - ds["lat"][0])) if ds.sizes.get("lat", 0) > 1 else None
    lon_res = abs(float(ds["lon"][1] - ds["lon"][0])) if ds.sizes.get("lon", 0) > 1 else None
    if lat_res and lon_res and abs(lat_res - 0.125) < 0.02 and abs(lon_res - 0.125) < 0.02 \
            and ds.sizes["lat"] % 2 == 0 and ds.sizes["lon"] % 2 == 0:
        return ds.coarsen(lat=2, lon=2, boundary="exact").mean(keep_attrs=True)
    print(f"  ⚠️ native step {lat_res}/{lon_res}° not 0.125°-even; interpolating to 0.25° grid.")
    return ds.interp(lat=np.arange(REGION["min_lat"], REGION["max_lat"] + 0.001, 0.25),
                     lon=np.arange(REGION["min_lon"], REGION["max_lon"] + 0.001, 0.25),
                     method="linear", kwargs={"fill_value": "extrapolate"})


def download_sss_month(year, month, username, password, outdir=OUTPUT_ROOT / "CMEMS_SSS"):
    """Download daily 0.25° SSS for a month (MY+NRT stitch, 1993→present).

    Product: MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013 (daily gap-free L4).
    Region: 5°N–30°N, 45°E–105°E. One output file per available day:
    CMEMS_SSS/YYYY/MM/sss_YYYYMMDD.nc (variable `sss`, 0.25° grid).
    """
    import pandas as pd

    n_days = _days_in_month(year, month)
    print(f"\n{'=' * 60}\nDownloading CMEMS SSS: {year}-{month:02d}\n"
          f"Product: {SSS_PRODUCT_ID}\nResolution: 0.25° × 0.25° (regridded from 0.125° daily L4)\n"
          f"Region: {REGION['min_lat']}–{REGION['max_lat']}°N, "
          f"{REGION['min_lon']}–{REGION['max_lon']}°E\n{'=' * 60}")

    tmp = outdir / f"_tmp_sss_{year}{month:02d}.nc"
    tmp.parent.mkdir(parents=True, exist_ok=True)
    if tmp.exists():
        tmp.unlink()

    dataset_used, last_err = None, None
    for dataset_id in _sss_candidate_datasets(year, month):
        try:
            print(f"  Trying dataset: {dataset_id}")
            _sss_download_subset(dataset_id, year, month, n_days, username, password, tmp)
            dataset_used = dataset_id
            break
        except Exception as e:
            last_err = e
            print(f"  ⚠️ {dataset_id} failed: {type(e).__name__}: {e}")
            if tmp.exists():
                tmp.unlink()
    if dataset_used is None:
        print(f"❌ CMEMS SSS download failed for {year}-{month:02d}: {last_err}")
        return
    print(f"  Dataset used: {dataset_used}")

    try:
        ds = xr.open_dataset(tmp)
        print(f"  Raw dimensions: {dict(ds.sizes)} | vars: {list(ds.data_vars)}")
        ds, sss_var = _sss_normalize(ds)
        if "lat" in ds.coords and float(ds.lat[0]) > float(ds.lat[-1]):
            ds = ds.sortby("lat")
        if "lon" in ds.coords and float(ds.lon[0]) > float(ds.lon[-1]):
            ds = ds.sortby("lon")

        ds = ds.sel(lat=slice(REGION["min_lat"], REGION["max_lat"]),
                      lon=slice(REGION["min_lon"], REGION["max_lon"]))
        if ds.sizes.get("lat", 0) == 0 or ds.sizes.get("lon", 0) == 0:
            raise ValueError("Spatial subset is empty.")
        print(f"  Native spatial grid: {ds.sizes['lat']} × {ds.sizes['lon']}")
        ds = _sss_regrid_025(ds)
        lat_res = abs(float(ds["lat"][1] - ds["lat"][0]))
        lon_res = abs(float(ds["lon"][1] - ds["lon"][0]))
        print(f"  Regridded: {ds.sizes['lat']} × {ds.sizes['lon']} @ {lat_res:.4f}° × {lon_res:.4f}°")
        if abs(lat_res - 0.25) > 0.02 or abs(lon_res - 0.25) > 0.02:
            print("  ⚠️ WARNING: output grid is not 0.25°!")
        if "time" not in ds.coords:
            raise ValueError("Dataset does not contain a time coordinate.")
        try:
            times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
        except Exception:
            times = pd.DatetimeIndex(ds.indexes["time"].to_datetimeindex())
        print(f"  Available daily observations: {len(times)}")
        n_saved = 0
        for ts in times:
            d_str = ts.strftime("%Y%m%d")
            date_str = ts.strftime("%Y-%m-%d")
            outfile = outdir / f"{year}" / f"{month:02d}" / f"sss_{d_str}.nc"
            if outfile.exists():
                print(f"  ⏭️ Exists: {outfile.name}")
                continue
            outfile.parent.mkdir(parents=True, exist_ok=True)
            try:
                daily = ds.sel(time=str(date_str))
                if "depth" in daily.dims:
                    daily = daily.isel(depth=0, drop=True)
                daily = daily[[sss_var]].rename({sss_var: "sss"})
                daily.attrs["source_product"] = SSS_PRODUCT_ID
                daily.attrs["source_dataset"] = dataset_used
                daily.to_netcdf(outfile, format="NETCDF4")
                n_saved += 1
                print(f"  ✅ {date_str} → {outfile.name}")
            except Exception as e:
                print(f"  ⚠️ Skip {date_str}: {e}")
        ds.close()
        print(f"\n✅ CMEMS SSS {year}-{month:02d} complete ({n_saved} new) → {outdir}")
    except Exception as e:
        print(f"❌ Error processing {year}-{month:02d}: {type(e).__name__}: {e}")
    finally:
        if tmp.exists():
            tmp.unlink()


In [8]:
import os
from pathlib import Path

# Replace with your actual key if this env var isn't already set in this session
CDS_API_KEY = os.environ.get("CDS_API_KEY")
if not CDS_API_KEY:
    raise RuntimeError("Set CDS_API_KEY first (Colab Secrets or the Credentials cell).")

cdsapirc_path = Path.home() / ".cdsapirc"
cdsapirc_path.write_text(f"url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n")

os.environ["CDS_API_KEY"] = CDS_API_KEY  # keep env var set too, other code may reference it

print("File written to:", cdsapirc_path)
print("Exists:", cdsapirc_path.exists())
print("Content:\n", cdsapirc_path.read_text().replace(CDS_API_KEY, "***HIDDEN***"))

File written to: /root/.cdsapirc
Exists: True
Content:
 url: https://cds.climate.copernicus.eu/api
key: ***HIDDEN***



In [9]:
import shutil
from pathlib import Path

era5_dir = OUTPUT_ROOT / "ERA5_WINDS"

if era5_dir.exists():
    shutil.rmtree(era5_dir)
    print(f"Deleted: {era5_dir}")
else:
    print("ERA5 folder does not exist.")

era5_dir.mkdir(parents=True, exist_ok=True)

print("Fresh ERA5 folder created.")

Deleted: /content/ocean_data/ERA5_WINDS
Fresh ERA5 folder created.


In [10]:
import cdsapi
import numpy as np
import xarray as xr
from pathlib import Path
import calendar


# ============================================================
# COMMON GRID
# ============================================================

TARGET_LAT = np.arange(5.125, 30.0, 0.25)
TARGET_LON = np.arange(45.125, 105.0, 0.25)

print("Target grid:")
print("Latitude points :", len(TARGET_LAT))
print("Longitude points:", len(TARGET_LON))


# ============================================================
# DAYS IN MONTH
# ============================================================

def _days_in_month(year, month):
    return calendar.monthrange(year, month)[1]


# ============================================================
# DOWNLOAD + PROCESS ONE MONTH
# ============================================================

def download_era5_month(year, month, outdir=OUTPUT_ROOT / "ERA5_WINDS"):

    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    n_days = _days_in_month(year, month)

    tmp = outdir / f"_tmp_era5_{year}{month:02d}.nc"

    print(f"\nDownloading ERA5: {year}-{month:02d}")

    c = cdsapi.Client()

    # --------------------------------------------------------
    # DOWNLOAD RAW MONTHLY DATA
    # --------------------------------------------------------

    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",

            "variable": [
                "10m_u_component_of_wind",
                "10m_v_component_of_wind"
            ],

            "year": str(year),
            "month": f"{month:02d}",

            "day": [
                f"{d:02d}"
                for d in range(1, n_days + 1)
            ],

            # Four values per day
            "time": [
                "00:00",
                "06:00",
                "12:00",
                "18:00"
            ],

            # Larger boundary region
            # This allows interpolation to 5.125 etc.
            "area": [
                30.0,   # North
                45.0,   # West
                5.0,    # South
                105.0   # East
            ],

            "data_format": "netcdf",
        },

        str(tmp)
    )

    print("Download complete.")


    # ========================================================
    # OPEN DATASET
    # ========================================================

    ds = xr.open_dataset(tmp)


    # Fix time dimension name if necessary
    if "valid_time" in ds.dims:
        ds = ds.rename({"valid_time": "time"})


    # ========================================================
    # DAILY MEAN
    # ========================================================

    ds_daily = ds.resample(
        time="1D"
    ).mean()


    # ========================================================
    # ENSURE LATITUDE IS ASCENDING
    # ========================================================

    if ds_daily.latitude.values[0] > ds_daily.latitude.values[-1]:

        ds_daily = ds_daily.sortby(
            "latitude"
        )


    # ========================================================
    # INTERPOLATE TO COMMON GRID
    # ========================================================

    print("Interpolating to common grid...")

    ds_daily = ds_daily.interp(
        latitude=TARGET_LAT,
        longitude=TARGET_LON,
        method="linear"
    )


    # ========================================================
    # WIND SPEED
    # ========================================================

    ds_daily["wind_speed"] = np.sqrt(
        ds_daily["u10"] ** 2 +
        ds_daily["v10"] ** 2
    )


    # ========================================================
    # SAVE EACH DAY
    # ========================================================

    for day in range(1, n_days + 1):

        date_str = f"{year}-{month:02d}-{day:02d}"

        outfile = (
            outdir /
            str(year) /
            f"{month:02d}" /
            f"era5_winds_{year}{month:02d}{day:02d}.nc"
        )

        outfile.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        daily_data = ds_daily.sel(
            time=date_str,
            method="nearest"
        )

        daily_data.to_netcdf(outfile)

        daily_data.close()


    # ========================================================
    # CLEANUP
    # ========================================================

    ds.close()
    ds_daily.close()

    tmp.unlink()

    print(
        f"Completed ERA5 {year}-{month:02d}"
    )

Target grid:
Latitude points : 100
Longitude points: 240


In [11]:
# ── 4d. NOAA OISST AVHRR v2.1 SST ───────────────────────────────────────────
# Direct NCEI per-day files (0.25° global, lon 0–360°):
#   .../v2.1/access/avhrr/YYYYMM/oisst-avhrr-v02r01.YYYYMMDD.nc
# Days NCEI has not published yet (~2–4 wk lag) 404 → skipped with a note.
# Output: NOAA_OISST/YYYY/MM/oisst_YYYYMMDD.nc (sst, regional 0.25°).
# Stdlib download (urllib); xarray only for the regional subset.

import urllib.error
import urllib.request

OISST_BASE = ("https://www.ncei.noaa.gov/data/sea-surface-temperature-"
              "optimum-interpolation/v2.1/access/avhrr")


def _oisst_url(year, month, day):
    return (f"{OISST_BASE}/{year}{month:02d}/"
            f"oisst-avhrr-v02r01.{year}{month:02d}{day:02d}.nc")


def download_oisst_month(year, month, outdir=OUTPUT_ROOT / "NOAA_OISST"):
    import xarray as _xr
    n_days = _days_in_month(year, month)
    n_saved, n_skip, n_missing = 0, 0, 0
    for day in range(1, n_days + 1):
        date_str = f"{year}-{month:02d}-{day:02d}"
        outfile = (outdir / f"{year}" / f"{month:02d}"
                   / f"oisst_{year}{month:02d}{day:02d}.nc")
        if outfile.exists():
            continue
        outfile.parent.mkdir(parents=True, exist_ok=True)
        tmp_file = outfile.with_suffix(".tmp.nc")
        try:
            req = urllib.request.Request(
                _oisst_url(year, month, day),
                headers={"User-Agent": "ocean_downloader"})
            with urllib.request.urlopen(req, timeout=120) as resp, \
                    open(tmp_file, "wb") as fh:
                fh.write(resp.read())
        except urllib.error.HTTPError as e:
            if e.code == 404:
                n_missing += 1
                continue
            print(f"  skip {date_str}: HTTP {e.code}")
            n_skip += 1
            continue
        except Exception as e:
            print(f"  skip {date_str}: {e}")
            n_skip += 1
            continue
        try:
            ds = _xr.open_dataset(tmp_file)
            sub = ds[["sst"]].isel(time=0, zlev=0).sel(
                lat=slice(REGION["min_lat"], REGION["max_lat"]),
                lon=slice(REGION["min_lon"], REGION["max_lon"]))
            sub.encoding.pop("unlimited_dims", None)  # source marks time unlimited
            sub.to_netcdf(outfile)
            ds.close()
            n_saved += 1
        except Exception as e:
            print(f"  skip {date_str}: {e}")
            n_skip += 1
            if outfile.exists():
                outfile.unlink()
        finally:
            if tmp_file.exists():
                tmp_file.unlink()
    print(f"✅ OISST {year}-{month:02d} done ({n_saved} new, {n_skip} skipped, "
          f"{n_missing} not-yet-published) → {outdir}")


---
## 5 — Run All Four Datasets (convenience)

In [12]:
user, pwd = load_cmems_credentials()

from datetime import date

START = date(2025, 1, 1)
END   = date(2025, 3, 1)

y, m = START.year, START.month

while (y, m) <= (END.year, END.month):

    download_altimetry_month(
        y, m, user, pwd
    )
    m += 1

    if m > 12:
        m = 1
        y += 1

print("Altimetry data done")

INFO - 2026-09-09T10:23:54Z - Selected dataset version: "202506"
INFO:copernicusmarine:Selected dataset version: "202506"
INFO - 2026-09-09T10:23:54Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-09-09T10:24:29Z - Total size of the download: 68.17 MB.
INFO:copernicusmarine:Total size of the download: 68.17 MB.


✅ Altimetry 2025-01 done (0 new, cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D) → /content/ocean_data/CMEMS_ALTIMETRY


INFO - 2026-09-09T10:24:32Z - Selected dataset version: "202506"
INFO:copernicusmarine:Selected dataset version: "202506"
INFO - 2026-09-09T10:24:32Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


...

✅ Altimetry 2025-02 done (0 new, cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D) → /content/ocean_data/CMEMS_ALTIMETRY


INFO - 2026-09-09T10:25:23Z - Selected dataset version: "202506"
INFO:copernicusmarine:Selected dataset version: "202506"
INFO - 2026-09-09T10:25:23Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-09-09T10:25:54Z - Total size of the download: 68.17 MB.
INFO:copernicusmarine:Total size of the download: 68.17 MB.


✅ Altimetry 2025-03 done (0 new, cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.125deg_P1D) → /content/ocean_data/CMEMS_ALTIMETRY
Altimetry data done


In [13]:
from datetime import date

START = date(2025, 1, 1)
END   = date(2025, 3, 1)

y, m = START.year, START.month

while (y, m) <= (END.year, END.month):

    download_sss_month(
        y, m, user, pwd
    )
    m += 1

    if m > 12:
        m = 1
        y += 1

if m == 1:
    last_y, last_m = y - 1, 12
else:
    last_y, last_m = y, m - 1
print(f"✅ SSS {last_y}-{last_m:02d} done.")


Product: MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013
Resolution: 0.25° × 0.25° (regridded from 0.125° daily L4)
Region: 5.0–30.0°N, 45.0–105.0°E
  Trying dataset: cmems_obs-mob_glo_phy-sss_nrt_multi_P1D


INFO - 2026-09-09T10:25:57Z - Selected dataset version: "202607"
INFO:copernicusmarine:Selected dataset version: "202607"
INFO - 2026-09-09T10:25:57Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-09-09T10:26:05Z - Total size of the download: 11.37 MB.
INFO:copernicusmarine:Total size of the download: 11.37 MB.


  Dataset used: cmems_obs-mob_glo_phy-sss_nrt_multi_P1D
  Raw dimensions: {'time': 31, 'depth': 1, 'latitude': 200, 'longitude': 480} | vars: ['sos']
  Native spatial grid: 200 × 480
  Regridded: 100 × 240 @ 0.2500° × 0.2500°
  Available daily observations: 31
  ⏭️ Exists: sss_20250101.nc
  ⏭️ Exists: sss_20250102.nc
  ⏭️ Exists: sss_20250103.nc
  ⏭️ Exists: sss_20250104.nc
  ⏭️ Exists: sss_20250105.nc
  ⏭️ Exists: sss_20250106.nc
  ⏭️ Exists: sss_20250107.nc
  ⏭️ Exists: sss_20250108.nc
  ⏭️ Exists: sss_20250109.nc
  ⏭️ Exists: sss_20250110.nc
  ⏭️ Exists: sss_20250111.nc
  ⏭️ Exists: sss_20250112.nc
  ⏭️ Exists: sss_20250113.nc
  ⏭️ Exists: sss_20250114.nc
  ⏭️ Exists: sss_20250115.nc
  ⏭️ Exists: sss_20250116.nc
  ⏭️ Exists: sss_20250117.nc
  ⏭️ Exists: sss_20250118.nc
  ⏭️ Exists: sss_20250119.nc
  ⏭️ Exists: sss_20250120.nc
  ⏭️ Exists: sss_20250121.nc
  ⏭️ Exists: sss_20250122.nc
  ⏭️ Exists: sss_20250123.nc
  ⏭️ Exists: sss_20250124.nc
  ⏭️ Exists: sss_20250125.nc
  ⏭️ Exists: s

...

  Dataset used: cmems_obs-mob_glo_phy-sss_nrt_multi_P1D
  Raw dimensions: {'time': 28, 'depth': 1, 'latitude': 200, 'longitude': 480} | vars: ['sos']
  Native spatial grid: 200 × 480
  Regridded: 100 × 240 @ 0.2500° × 0.2500°
  Available daily observations: 28
  ⏭️ Exists: sss_20250201.nc
  ⏭️ Exists: sss_20250202.nc
  ⏭️ Exists: sss_20250203.nc
  ⏭️ Exists: sss_20250204.nc
  ⏭️ Exists: sss_20250205.nc
  ⏭️ Exists: sss_20250206.nc
  ⏭️ Exists: sss_20250207.nc
  ⏭️ Exists: sss_20250208.nc
  ⏭️ Exists: sss_20250209.nc
  ⏭️ Exists: sss_20250210.nc
  ⏭️ Exists: sss_20250211.nc
  ⏭️ Exists: sss_20250212.nc
  ⏭️ Exists: sss_20250213.nc
  ⏭️ Exists: sss_20250214.nc
  ⏭️ Exists: sss_20250215.nc
  ⏭️ Exists: sss_20250216.nc
  ⏭️ Exists: sss_20250217.nc
  ⏭️ Exists: sss_20250218.nc
  ⏭️ Exists: sss_20250219.nc
  ⏭️ Exists: sss_20250220.nc
  ⏭️ Exists: sss_20250221.nc
  ⏭️ Exists: sss_20250222.nc
  ⏭️ Exists: sss_20250223.nc
  ⏭️ Exists: sss_20250224.nc
  ⏭️ Exists: sss_20250225.nc
  ⏭️ Exists: s

INFO - 2026-09-09T10:26:19Z - Selected dataset version: "202607"
INFO:copernicusmarine:Selected dataset version: "202607"
INFO - 2026-09-09T10:26:19Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-09-09T10:26:27Z - Total size of the download: 11.37 MB.
INFO:copernicusmarine:Total size of the download: 11.37 MB.


  Dataset used: cmems_obs-mob_glo_phy-sss_nrt_multi_P1D
  Raw dimensions: {'time': 31, 'depth': 1, 'latitude': 200, 'longitude': 480} | vars: ['sos']
  Native spatial grid: 200 × 480
  Regridded: 100 × 240 @ 0.2500° × 0.2500°
  Available daily observations: 31
  ⏭️ Exists: sss_20250301.nc
  ⏭️ Exists: sss_20250302.nc
  ⏭️ Exists: sss_20250303.nc
  ⏭️ Exists: sss_20250304.nc
  ⏭️ Exists: sss_20250305.nc
  ⏭️ Exists: sss_20250306.nc
  ⏭️ Exists: sss_20250307.nc
  ⏭️ Exists: sss_20250308.nc
  ⏭️ Exists: sss_20250309.nc
  ⏭️ Exists: sss_20250310.nc
  ⏭️ Exists: sss_20250311.nc
  ⏭️ Exists: sss_20250312.nc
  ⏭️ Exists: sss_20250313.nc
  ⏭️ Exists: sss_20250314.nc
  ⏭️ Exists: sss_20250315.nc
  ⏭️ Exists: sss_20250316.nc
  ⏭️ Exists: sss_20250317.nc
  ⏭️ Exists: sss_20250318.nc
  ⏭️ Exists: sss_20250319.nc
  ⏭️ Exists: sss_20250320.nc
  ⏭️ Exists: sss_20250321.nc
  ⏭️ Exists: sss_20250322.nc
  ⏭️ Exists: sss_20250323.nc
  ⏭️ Exists: sss_20250324.nc
  ⏭️ Exists: sss_20250325.nc
  ⏭️ Exists: s

In [14]:
from datetime import date

START = date(2025, 1, 1)
END   = date(2025, 3, 1)

y, m = START.year, START.month

while (y, m) <= (END.year, END.month):

    download_era5_month(
        y, m
    )
    m += 1

    if m > 12:
        m = 1
        y += 1

print("ERA 5 Done")

2026-09-09 10:26:29,164 INFO Request ID is 2659fa26-166a-40d5-a4ca-bec680bc71ee
INFO:ecmwf.datastores.legacy_client:Request ID is 2659fa26-166a-40d5-a4ca-bec680bc71ee
2026-09-09 10:26:29,489 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-09 10:26:53,508 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-09 10:27:05,085 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


9595cee0b59fe4e810bfbdbac254d954.nc:   0%|          | 0.00/12.4M [00:00<?, ?B/s]

Download complete.
Interpolating to common grid...
Completed ERA5 2025-01



2026-09-09 10:27:11,648 INFO Request ID is 05905675-bd67-4a96-a164-f110afb6231d
INFO:ecmwf.datastores.legacy_client:Request ID is 05905675-bd67-4a96-a164-f110afb6231d
2026-09-09 10:27:11,843 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-09 10:28:07,345 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


921c4a1b32a782e260acb25d1709a5cf.nc:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

Download complete.
Interpolating to common grid...
Completed ERA5 2025-02



2026-09-09 10:28:14,071 INFO Request ID is a49c7c68-f42f-403c-ab3b-e4da50ff439c
INFO:ecmwf.datastores.legacy_client:Request ID is a49c7c68-f42f-403c-ab3b-e4da50ff439c
2026-09-09 10:28:14,266 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-09 10:28:31,984 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-09 10:28:39,772 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


f5ceb99676129fee30d559e3ce1a1df7.nc:   0%|          | 0.00/12.5M [00:00<?, ?B/s]

Download complete.
Interpolating to common grid...
Completed ERA5 2025-03
ERA 5 Done


In [15]:
from datetime import date

START = date(2025, 1, 1)
END   = date(2025, 3, 1)

y, m = START.year, START.month

while (y, m) <= (END.year, END.month):

    download_oisst_month(
        y, m
    )

    m += 1

    if m > 12:
        m = 1
        y += 1

print("✨ All datasets downloaded for the range.")

✅ OISST 2025-01 done (31 new, 0 skipped, 0 not-yet-published) → /content/ocean_data/NOAA_OISST
✅ OISST 2025-02 done (28 new, 0 skipped, 0 not-yet-published) → /content/ocean_data/NOAA_OISST
✅ OISST 2025-03 done (31 new, 0 skipped, 0 not-yet-published) → /content/ocean_data/NOAA_OISST
✨ All datasets downloaded for the range.


---
## 6 - Multi-Model Comparative Analysis

Predict 3D ocean temperature profiles (0-5500 m) using **5 architectures**:

| Model | Architecture | Key feature |
|---|---|---|
| **U-Net** | Encoder-decoder | 4-scale skip connections |
| **ResNet** | Fully-conv residual | No pooling, full resolution |
| **Attention U-Net** | U-Net + attention gates | Learned skip weighting |
| **DenseNet** | Dense feature reuse | All-to-all layer connections |
| **Simple CNN** | Multi-scale conv | Fastest training baseline |

**Region:** 5N-30N, 45E-105E | **Features:** 11 surface channels | **Target:** GLORYS temp at 15 depths

In [16]:
# -- Install ML dependencies --------------------------------------------------
%pip install -q scikit-learn xgboost torch torchvision
print('ML dependencies installed.')


ML dependencies installed.


### 6a. Download GLORYS12v1 Temperature Profiles

In [ ]:
# -- Self-contained (no external package needed): data lives under OUTPUT_ROOT --
def resolve_root(default="ocean_data"):
    return OUTPUT_ROOT

"""GLORYS12v1 ocean temperature profiles (thetao), MY+NRT stitch.

MY : GLOBAL_MULTIYEAR_PHY_001_030
NRT: GLOBAL_ANALYSISFORECAST_PHY_001_024

Native 1/12° × 50 depth levels → coarsened 3×3 to 0.25° here.
Output: <OUTDIR>/YYYY/MM/glorys_YYYYMMDD.nc with thetao(depth, lat, lon)
depths ascending (surface first), NetCDF4+compression.
"""

import calendar
import logging
from datetime import date
from pathlib import Path
import xarray as xr

log = logging.getLogger(__name__)

PRODUCT_MY = "GLOBAL_MULTIYEAR_PHY_001_030"
PRODUCT_NRT = "GLOBAL_ANALYSISFORECAST_PHY_001_024"
MY_DATASET = "cmems_mod_glo_phy_my_0.083deg_P1D-m"
NRT_DATASET = "cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m"
NRT_CUTOFF = (2026, 7)
VAR = "thetao"
REGION_DEFAULT = dict(min_lat=5.0, max_lat=30.0, min_lon=45.0, max_lon=105.0)

# ✅ Only keep these 15 depths
TARGET_DEPTHS = [0, 5, 10, 20, 30, 50, 75, 100,
                 150, 200, 300, 500, 700, 900, 1000]

START_DATE = date(2025, 1, 1)
END_DATE = date(2025, 3, 1)

OUTDIR = resolve_root() / "GLORYS_TEMP"

def setup_logging(level=logging.INFO):
    logging.basicConfig(level=level, format="%(asctime)s %(levelname)s %(message)s")

def _days_in_month(year, month):
    return calendar.monthrange(year, month)[1]

def outdir(root=None):
    return Path(root) if root else OUTDIR

def day_file(year, month, day, root=None):
    d = f"{year}{month:02d}{day:02d}"
    return outdir(root) / f"{year}" / f"{month:02d}" / f"glorys_{d}.nc"

def _month_complete(year, month, root=None):
    n = _days_in_month(year, month)
    d = outdir(root) / f"{year}" / f"{month:02d}"
    return all((d / f"glorys_{year}{month:02d}{day:02d}.nc").exists()
               for day in range(1, n + 1))

def candidate_datasets(year, month):
    if (year, month) < NRT_CUTOFF:
        return [MY_DATASET, NRT_DATASET]
    return [NRT_DATASET, MY_DATASET]

def download_month(year, month, username, password, root=None, region=None):
    import copernicusmarine
    region = REGION_DEFAULT
    n_days = _days_in_month(year, month)
    if _month_complete(year, month, root):
        return True
    tmp = outdir(root) / f"_tmp_glorys_{year}{month:02d}.nc"
    tmp.parent.mkdir(parents=True, exist_ok=True)
    if tmp.exists():
        tmp.unlink()

    dataset_used, last_err = None, None
    for dataset_id in candidate_datasets(year, month):
        try:
            copernicusmarine.subset(
                dataset_id=dataset_id, variables=[VAR],
                start_datetime=f"{year}-{month:02d}-01T00:00:00",
                end_datetime=f"{year}-{month:02d}-{n_days:02d}T23:59:59",
                minimum_latitude=region["min_lat"],
                maximum_latitude=region["max_lat"],
                minimum_longitude=region["min_lon"],
                maximum_longitude=region["max_lon"],
                output_filename=tmp.name, output_directory=str(tmp.parent),
                username=username, password=password, overwrite=True)
            dataset_used = dataset_id
            break
        except Exception as e:
            last_err = e
            log.warning("  %s failed for %04d-%02d: %s", dataset_id, year, month, e)
            if tmp.exists():
                tmp.unlink()
    if dataset_used is None:
        log.error("GLORYS %04d-%02d failed: %s", year, month, last_err)
        return _month_complete(year, month, root)

    try:
        ds = xr.open_dataset(tmp)
        if "latitude" in ds.coords:
            ds = ds.rename({"latitude": "lat"})
        if "longitude" in ds.coords:
            ds = ds.rename({"longitude": "lon"})
        ds = ds.sortby("lat")
        ds = ds.sortby("lon")
        if "depth" in ds.coords:
            ds = ds.sortby("depth")

        ds = ds.sel(lat=slice(region["min_lat"], region["max_lat"]),
                    lon=slice(region["min_lon"], region["max_lon"]))

        # coarsen to 0.25° grid
        import numpy as np
        if ds.sizes.get("lat", 0) % 3 == 0 and ds.sizes.get("lon", 0) % 3 == 0:
            ds = ds.coarsen(lat=3, lon=3, boundary="exact").mean(keep_attrs=True)
        else:
            ds = ds.interp(lat=TARGET_LAT, lon=TARGET_LON,
                            method="linear", kwargs={"fill_value": "extrapolate"})

        # ✅ Interpolate to the exact PS-specified depths (not nearest-neighbor snap)
        if "depth" in ds.coords:
            ds = ds.interp(depth=TARGET_DEPTHS, method="linear",
                           kwargs={"fill_value": "extrapolate"})

        import pandas as pd
        times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
        for ts in times:
            d_str, date_str = ts.strftime("%Y%m%d"), ts.strftime("%Y-%m-%d")
            outfile = day_file(ts.year, ts.month, ts.day, root)
            if outfile.exists():
                continue
            outfile.parent.mkdir(parents=True, exist_ok=True)
            daily = ds.sel(time=date_str)[[VAR]]
            daily.attrs["source_dataset"] = dataset_used
            daily.to_netcdf(outfile, format="NETCDF4",
                            encoding={VAR: {"zlib": True, "complevel": 4}})
        ds.close()
    finally:
        if tmp.exists():
            tmp.unlink()

    ok = _month_complete(year, month, root)
    log.info("GLORYS %04d-%02d %s (%s)", year, month,
             "complete" if ok else "INCOMPLETE", dataset_used)
    return ok

# -- Run the GLORYS download --
print('Starting GLORYS temperature download...')
setup_logging()
username, password = load_cmems_credentials()

current = date(START_DATE.year, START_DATE.month, 1)
failed = []
while current <= END_DATE:
    y, m = current.year, current.month
    if _month_complete(y, m):
        print(f'  {y:04d}-{m:02d}  already complete -- skipped')
    else:
        if not download_month(y, m, username, password):
            failed.append(f'{y:04d}-{m:02d}')
    m += 1
    if m > 12:
        m, y = 1, y + 1
    current = date(y, m, 1)

print('\n' + '=' * 50)
print(f'Failed: {", ".join(failed)}' if failed else 'GLORYS download complete.')
print(f'{OUTDIR}')


Starting GLORYS temperature download...


INFO - 2026-09-09T10:30:00Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-09T10:30:00Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-09-09T10:32:35Z - Total size of the download: 641.96 MB.
INFO:copernicusmarine:Total size of the download: 641.96 MB.
INFO - 2026-09-09T10:32:55Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-09T10:32:55Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-09-09T10:35:20Z - Total size of the download: 579.84 MB.
INFO:copernicusmarine:Total size of the download: 579.84 MB.
INFO - 2026-09-09T10:35:38Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-09T10:35:38Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-09-09T10:38:11Z - Total size of the download: 641.96 MB.
INFO:copernicusmarine:Total size of the download: 641.96 MB.



GLORYS download complete.
/content/ocean_data/GLORYS_TEMP


In [ ]:
from pathlib import Path
from datetime import date
import xarray as xr
import logging
import numpy as np

log = logging.getLogger(__name__)

def resolve_root(default="ocean_data"):
    return OUTPUT_ROOT

# ============================================================
# INCOIS Gridded ARGO (independent validation, not used for training)
# ============================================================
"""INCOIS ARGO Monthly Variational Analysis Methodology (gridded).

Source: https://erddap.incois.gov.in/erddap/griddap/incois_argo_mnt_VAM
Monthly resolution, TEMP variable, fill value -9999.0.
Used ONLY for independent validation -- never for training.
"""

ARGO_URL = "https://erddap.incois.gov.in/erddap/griddap/incois_argo_mnt_VAM.nc"
ARGO_OUTDIR = resolve_root() / "INCOIS_ARGO"

# Define START_DATE and END_DATE for the Argo download
START_DATE = date(2025, 1, 1)
END_DATE = date(2025, 3, 1)

# Define REGION_DEFAULT for the Argo download
REGION_DEFAULT = dict(min_lat=5.0, max_lat=30.0, min_lon=45.0, max_lon=105.0)

# Define TARGET_LAT and TARGET_DEPTHS for regridding (from other cells)
TARGET_LAT = np.arange(5.125, 30.0, 0.25)
TARGET_LON = np.arange(45.125, 105.0, 0.25)
TARGET_DEPTHS = [0, 5, 10, 20, 30, 50, 75, 100, 150, 200, 300, 500, 700, 900, 1000]


def _argo_month_file(year, month, root=None):
    d = Path(root) if root else ARGO_OUTDIR
    return d / f"argo_{year}{month:02d}.nc"


def download_argo_month(year, month, region=None, root=None):
    region = region or REGION_DEFAULT
    outfile = _argo_month_file(year, month, root)
    if outfile.exists():
        log.info("  Argo %04d-%02d already complete -- skipped", year, month)
        return True
    outfile.parent.mkdir(parents=True, exist_ok=True)

    date_str = f"{year}-{month:02d}-15T00:00:00Z"  # monthly data, mid-month timestamp
    query = (f"?TEMP[({date_str})][(5.0):(1000.0)]"
             f"[({region['min_lat']}):({region['max_lat']})]"
             f"[({region['min_lon']}):({region['max_lon']})]")
    try:
        ds = xr.open_dataset(ARGO_URL + query)
        ds.to_netcdf(outfile)
        ds.close()
        log.info("  Argo %04d-%02d saved", year, month)
        return True
    except Exception as e:
        log.warning("  Argo %04d-%02d failed: %s", year, month, e)
        return False


def load_argo_month_regridded(year, month, root=None):
    """Loads one month, masks fill values, regrids onto the SAME lat/lon/depth
    grid your model outputs -- so it's directly comparable, cell for cell."""
    f = _argo_month_file(year, month, root)
    if not f.exists():
        return None
    ds = xr.open_dataset(f)
    da = ds["TEMP"].where(ds["TEMP"] != -9999.0)  # mask fill value -> NaN
    if "ZAX" in da.dims:
        da = da.rename({"ZAX": "depth"})
    if "time" in da.dims:
        da = da.isel(time=0, drop=True)
    da = da.interp(latitude=TARGET_LAT, longitude=TARGET_LON,
                   depth=TARGET_DEPTHS, method="linear")
    ds.close()
    return da.values.astype("float32")  # (depth, H, W), NaN where no Argo coverage


# -- Run the Argo download for the same period as your GLORYS training data --
print('\nStarting INCOIS Argo download (validation only)...')
current = date(START_DATE.year, START_DATE.month, 1)
argo_failed = []
while current <= END_DATE:
    y, m = current.year, current.month
    if not download_argo_month(y, m):
        argo_failed.append(f'{y:04d}-{m:02d}')
    m += 1
    if m > 12:
        m, y = 1, y + 1
    current = date(y, m, 1)

print('=' * 50)
print(f'Argo failed: {", ".join(argo_failed)}' if argo_failed else 'Argo download complete.')
print(f'{ARGO_OUTDIR}')



Starting INCOIS Argo download (validation only)...
Argo failed: 2025-01, 2025-02, 2025-03
/content/ocean_data/INCOIS_ARGO


In [ ]:
from pathlib import Path
from datetime import date
import xarray as xr
import numpy as np

root = Path("ocean_data")
start = date(2025, 1, 1)
end   = date(2025, 3, 1)

folders = {
    "altimetry": "CMEMS_ALTIMETRY",
    "sss":       "CMEMS_SSS",
    "era5":      "ERA5_WINDS",
    "oisst":     "NOAA_OISST",
    "glorys":    "GLORYS_TEMP",
}

print("Checking coverage, spatial and temporal resolution...\n")
cur = start
while cur <= end:
    y, m = cur.year, cur.month
    print(f"{y}-{m:02d}")
    for name, folder in folders.items():
        month_dir = root / folder / f"{y}" / f"{m:02d}"
        files = sorted(month_dir.glob("*.nc"))
        res_str, time_str = "?", "?"
        if files:
            # spatial resolution from first file
            try:
                with xr.open_dataset(files[0]) as ds:
                    lat_name = "lat" if "lat" in ds.coords else "latitude"
                    lon_name = "lon" if "lon" in ds.coords else "longitude"
                    lat = np.array(ds[lat_name])
                    lon = np.array(ds[lon_name])
                    if len(lat) > 1 and len(lon) > 1:
                        dlat = abs(lat[1] - lat[0])
                        dlon = abs(lon[1] - lon[0])
                        res_str = f"{dlat:.3f}° × {dlon:.3f}°"
                    else:
                        res_str = "single point"
            except Exception as e:
                res_str = f"error: {e}"

            # temporal resolution inferred from filenames
            try:
                dates = [f.stem.split("_")[-1] for f in files]  # extract YYYYMMDD
                if len(dates) > 1:
                    diffs = np.diff([np.datetime64(d, 'D') for d in dates])
                    step_days = np.unique(diffs.astype(int))
                    if len(step_days) == 1:
                        time_str = f"{step_days[0]} day step"
                    else:
                        time_str = f"varies: {step_days}"
                else:
                    time_str = "single day file"
            except Exception as e:
                time_str = f"error: {e}"

        print(f"  {name:<10} → {len(files)} files, spatial {res_str}, temporal {time_str}")
    print()
    # advance one month
    m += 1
    if m > 12:
        m, y = 1, y + 1
    cur = date(y, m, 1)


Checking coverage, spatial and temporal resolution...

2025-01
  altimetry  → 31 files, spatial 0.250° × 0.250°, temporal varies: [365 366]
  sss        → 31 files, spatial 0.250° × 0.250°, temporal varies: [365 366]
  era5       → 31 files, spatial 0.250° × 0.250°, temporal varies: [365 366]
  oisst      → 31 files, spatial 0.250° × 0.250°, temporal varies: [365 366]
  glorys     → 31 files, spatial 0.250° × 0.250°, temporal varies: [365 366]

2025-02
  altimetry  → 28 files, spatial 0.250° × 0.250°, temporal varies: [365 366]
  sss        → 28 files, spatial 0.250° × 0.250°, temporal varies: [365 366]
  era5       → 28 files, spatial 0.250° × 0.250°, temporal varies: [365 366]
  oisst      → 28 files, spatial 0.250° × 0.250°, temporal varies: [365 366]
  glorys     → 28 files, spatial 0.250° × 0.250°, temporal varies: [365 366]

2025-03
  altimetry  → 31 files, spatial 0.250° × 0.250°, temporal varies: [365 366]
  sss        → 31 files, spatial 0.250° × 0.250°, temporal varies: [365 

In [ ]:
from datetime import date
import xarray as xr
from pathlib import Path

# Definition for resolve_root (copied from prepare-data cell)
def resolve_root():
    return Path(OUTPUT_ROOT)

# Definition for FILES (copied from prepare-data cell)
FILES = {
    "altimetry": ("CMEMS_ALTIMETRY", "altimetry_{d}.nc"),
    "sss": ("CMEMS_SSS", "sss_{d}.nc"),
    "era5": ("ERA5_WINDS", "era5_winds_{d}.nc"),
    "oisst": ("NOAA_OISST", "oisst_{d}.nc"),
    "glorys": ("GLORYS_TEMP", "glorys_{d}.nc"),
}

# Definition for _day_path (copied from prepare-data cell)
def _day_path(kind, day, root):
    folder, pattern = FILES[kind]
    d = day.strftime("%Y%m%d")

    return (
        root / folder /
        day.strftime("%Y") /
        day.strftime("%m") /
        pattern.format(d=d)
    )

day = date(2025, 1, 1)
root = resolve_root()

paths = {
    "Altimetry": _day_path("altimetry", day, root),
    "SSS": _day_path("sss", day, root),
    "ERA5": _day_path("era5", day, root),
    "OISST": _day_path("oisst", day, root),
    "GLORYS": _day_path("glorys", day, root)
}

variables = {
    "Altimetry": ["sla", "adt", "ugos", "vgos", "ugosa", "vgosa"],
    "SSS": ["sss"],
    "ERA5": ["u10", "v10"],
    "OISST": ["sst"],
    "GLORYS": ["thetao"]
}

for dataset, path in paths.items():

    print(f"\n{'='*50}")
    print(dataset)
    print(f"{'='*50}")

    with xr.open_dataset(path) as ds:

        print("Dimensions:")
        print(ds.sizes)

        for var in variables[dataset]:

            if var in ds:

                print(
                    f"{var}: "
                    f"shape={ds[var].shape}, "
                    f"dims={ds[var].dims}"
                )

        # Print coordinates
        print("\nCoordinates:")

        for coord in ["lat", "latitude", "lon", "longitude"]:

            if coord in ds.coords:

                values = ds[coord].values

                print(
                    f"{coord}: "
                    f"len={len(values)}, "
                    f"range=({values.min()}, {values.max()})"
                )



Altimetry
Dimensions:
Frozen({'latitude': 100, 'longitude': 240})
sla: shape=(100, 240), dims=('latitude', 'longitude')
adt: shape=(100, 240), dims=('latitude', 'longitude')
ugos: shape=(100, 240), dims=('latitude', 'longitude')
vgos: shape=(100, 240), dims=('latitude', 'longitude')
ugosa: shape=(100, 240), dims=('latitude', 'longitude')
vgosa: shape=(100, 240), dims=('latitude', 'longitude')

Coordinates:
latitude: len=100, range=(5.125, 29.875)
longitude: len=240, range=(45.125, 104.875)

SSS
Dimensions:
Frozen({'lat': 100, 'lon': 240})
sss: shape=(100, 240), dims=('lat', 'lon')

Coordinates:
lat: len=100, range=(5.125, 29.875)
lon: len=240, range=(45.125, 104.875)

ERA5
Dimensions:
Frozen({'latitude': 100, 'longitude': 240})
u10: shape=(100, 240), dims=('latitude', 'longitude')
v10: shape=(100, 240), dims=('latitude', 'longitude')

Coordinates:
latitude: len=100, range=(5.125, 29.875)
longitude: len=240, range=(45.125, 104.875)

OISST
Dimensions:
Frozen({'lat': 100, 'lon': 240})
ss

### 6b. Prepare Training Data

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
from datetime import timedelta

# ============================================================
# CONFIG
# ============================================================

CHANNELS = [
    "sla", "adt", "ugos", "vgos", "ugosa", "vgosa",
    "sss", "u10", "v10", "wind_speed", "sst"
]

FILES = {
    "altimetry": ("CMEMS_ALTIMETRY", "altimetry_{d}.nc"),
    "sss": ("CMEMS_SSS", "sss_{d}.nc"),
    "era5": ("ERA5_WINDS", "era5_winds_{d}.nc"),
    "oisst": ("NOAA_OISST", "oisst_{d}.nc"),
    "glorys": ("GLORYS_TEMP", "glorys_{d}.nc"),
}


# ============================================================
# HELPERS
# ============================================================

def resolve_root():
    return Path(OUTPUT_ROOT)


def ml_dir(root):
    d = root / "ML_DATA"
    d.mkdir(parents=True, exist_ok=True)
    return d


def _date_range(start, end):
    days = []
    while start <= end:
        days.append(start)
        start += timedelta(days=1)
    return days


def _day_path(kind, day, root):
    folder, pattern = FILES[kind]
    d = day.strftime("%Y%m%d")

    return (
        root / folder /
        day.strftime("%Y") /
        day.strftime("%m") /
        pattern.format(d=d)
    )


def _clean(da):
    for dim in ["time", "valid_time", "zlev", "depth", "altitude"]:
        if dim in da.dims:
            da = da.isel({dim: 0}, drop=True)

    return da.values.astype(np.float32)


# ============================================================
# LOAD SURFACE DATA
# ============================================================

def _load_surface_day(day, root):

    paths = {
        "alt": _day_path("altimetry", day, root),
        "sss": _day_path("sss", day, root),
        "era5": _day_path("era5", day, root),
        "oisst": _day_path("oisst", day, root)
    }

    if not all(p.exists() for p in paths.values()):
        return None

    try:
        with (
            xr.open_dataset(paths["alt"]) as alt,
            xr.open_dataset(paths["sss"]) as sss,
            xr.open_dataset(paths["era5"]) as era5,
            xr.open_dataset(paths["oisst"]) as oisst
        ):

            alt_vars = ["sla", "adt", "ugos", "vgos", "ugosa", "vgosa"]

            if (
                not all(v in alt for v in alt_vars)
                or "sss" not in sss
                or "u10" not in era5
                or "v10" not in era5
                or "sst" not in oisst
            ):
                return None

            chans = [_clean(alt[v]) for v in alt_vars]

            chans.append(_clean(sss["sss"]))

            u10 = _clean(era5["u10"])
            v10 = _clean(era5["v10"])

            chans.extend([
                u10,
                v10,
                np.sqrt(u10**2 + v10**2).astype(np.float32),
                _clean(oisst["sst"])
            ])

            # All channels must have same H × W
            if len(set(x.shape for x in chans)) != 1:
                print(f"Shape mismatch on {day}")
                return None

            return np.stack(chans)

    except Exception as e:
        print(f"Surface error {day}: {e}")
        return None


# ============================================================
# LOAD GLORYS
# ============================================================

def _load_glorys_day(day, root):

    p = _day_path("glorys", day, root)

    if not p.exists():
        return None, None

    try:
        with xr.open_dataset(p) as ds:

            if "thetao" not in ds:
                return None, None

            da = ds["thetao"]

            if "time" in da.dims:
                da = da.isel(time=0, drop=True)

            return (
                da.values.astype(np.float32),
                da["depth"].values.astype(np.float32)
            )

    except Exception as e:
        print(f"GLORYS error {day}: {e}")
        return None, None


# ============================================================
# MAIN DATASET PREPARATION
# ============================================================

def prepare_cnn_dataset(
    start_date,
    end_date,
    root=None,
    day_stride=1,
    force=False
):

    root = Path(root) if root else resolve_root()

    cache = ml_dir(root) / (
        f"cnn_{start_date:%Y%m%d}_{end_date:%Y%m%d}.npz"
    )

    # Load existing dataset
    if cache.exists() and not force:
        with np.load(cache) as data:
            return {k: data[k] for k in data.files}

    X_list, Y_list, depths = [], [], None

    for i, day in enumerate(_date_range(start_date, end_date)):

        print(f"[{i+1}] Processing {day}")

        X = _load_surface_day(day, root)
        Y, d = _load_glorys_day(day, root)

        if X is None or Y is None:
            print("Skipping")
            continue

        # Check X and Y have same spatial grid shape
        if X.shape[1:] != Y.shape[1:]:
            print(
                f"Skipping {day}: "
                f"X {X.shape}, Y {Y.shape}"
            )
            continue

        X_list.append(X)
        Y_list.append(Y)
        depths = d

    if not X_list:
        raise ValueError("No valid days found")

    # --------------------------------------------------------
    # CREATE FINAL ARRAYS
    # --------------------------------------------------------

    Xg = np.stack(X_list).astype(np.float32)
    Yg = np.stack(Y_list).astype(np.float32)

    # Apply day stride
    Xg = Xg[::day_stride]
    Yg = Yg[::day_stride]

    n = len(Xg)
    i1, i2 = int(n * 0.8), int(n * 0.9)

    # --------------------------------------------------------
    # NORMALIZATION STATISTICS FROM TRAINING DATA ONLY
    # --------------------------------------------------------

    Y_mean = np.nanmean(
        Yg[:i1],
        axis=(0, 2, 3)
    ).astype(np.float32)

    Y_std = np.nanstd(
        Yg[:i1],
        axis=(0, 2, 3)
    ).astype(np.float32)

    Y_std[Y_std == 0] = 1.0

    # --------------------------------------------------------
    # CREATE SPLITS
    # --------------------------------------------------------

    out = {
        "channels": np.array(CHANNELS),
        "depths": depths,
        "Y_mean": Y_mean,
        "Y_std": Y_std,
        "H": np.int64(Xg.shape[2]),
        "W": np.int64(Xg.shape[3]),
        "n_days_kept": np.int64(n)
    }

    splits = {
        "train": (0, i1),
        "val": (i1, i2),
        "test": (i2, n)
    }

    for name, (a, b) in splits.items():

        X = Xg[a:b]

        Y = (
            Yg[a:b]
            - Y_mean[None, :, None, None]
        ) / Y_std[None, :, None, None]

        out[f"Xg_{name}"] = np.nan_to_num(X).astype(np.float32)
        out[f"Yg_{name}"] = np.nan_to_num(Y).astype(np.float32)

        print(
            f"{name}: "
            f"X {out[f'Xg_{name}'].shape}, "
            f"Y {out[f'Yg_{name}'].shape}"
        )

    np.savez_compressed(cache, **out)

    print(f"\nSaved to: {cache}")

    return out

In [ ]:
# -- PyTorch Dataset wrapper + demo --
import torch
from torch.utils.data import Dataset, DataLoader
from datetime import date
import logging

class OceanDataset(Dataset):
    def __init__(self, data, split="train"):
        self.X = torch.from_numpy(data[f"Xg_{split}"])
        self.Y = torch.from_numpy(data[f"Yg_{split}"])

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    # Build dataset splits
    data = prepare_cnn_dataset(
        start_date=date(2025, 1, 1),
        end_date=date(2025, 3, 1),
    )

    print("\nDataset ready!")
    print(f'  Xg_train: {data["Xg_train"].shape}  # (n_train, 11, H, W)')
    print(f'  Yg_train: {data["Yg_train"].shape}  # (n_train, D, H, W)')
    print(f'  Xg_val:   {data["Xg_val"].shape}')
    print(f'  Yg_val:   {data["Yg_val"].shape}')
    print(f'  Xg_test:  {data["Xg_test"].shape}')
    print(f'  Yg_test:  {data["Yg_test"].shape}')
    print(f'  Depths:   {len(data["depths"])} levels')

    # Example PyTorch loader
    train_ds = OceanDataset(data, split="train")
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)

    # Show a couple of batches
    for batch_idx, (X, Y) in enumerate(train_loader):
        print(f"Batch {batch_idx}: X {X.shape}, Y {Y.shape}")
        if batch_idx == 1:  # stop after two batches
            break


[1] Processing 2025-01-01
[2] Processing 2025-01-02
[3] Processing 2025-01-03
[4] Processing 2025-01-04
[5] Processing 2025-01-05
[6] Processing 2025-01-06
[7] Processing 2025-01-07
[8] Processing 2025-01-08
[9] Processing 2025-01-09
[10] Processing 2025-01-10
[11] Processing 2025-01-11
[12] Processing 2025-01-12
[13] Processing 2025-01-13
[14] Processing 2025-01-14
[15] Processing 2025-01-15
[16] Processing 2025-01-16
[17] Processing 2025-01-17
[18] Processing 2025-01-18
[19] Processing 2025-01-19
[20] Processing 2025-01-20
[21] Processing 2025-01-21
[22] Processing 2025-01-22
[23] Processing 2025-01-23
[24] Processing 2025-01-24
[25] Processing 2025-01-25
[26] Processing 2025-01-26
[27] Processing 2025-01-27
[28] Processing 2025-01-28
[29] Processing 2025-01-29
[30] Processing 2025-01-30
[31] Processing 2025-01-31
[32] Processing 2025-02-01
[33] Processing 2025-02-02
[34] Processing 2025-02-03
[35] Processing 2025-02-04
[36] Processing 2025-02-05
[37] Processing 2025-02-06
[38] Proce

In [ ]:
from google.colab import files
files.download("/content/ocean_data/ML_DATA/cnn_20250101_20250301.npz")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 6d. Train Deep Learning Models (U-Net, ResNet, Attention U-Net, DenseNet, Simple CNN)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ---------------- 1. LOAD DATA ----------------

DATA_PATH = "/content/ocean_data/ML_DATA/cnn_20250101_20250301.npz"
data = np.load(DATA_PATH)

X_train, Y_train = data["Xg_train"], data["Yg_train"]
X_val,   Y_val   = data["Xg_val"],   data["Yg_val"]
X_test,  Y_test  = data["Xg_test"],  data["Yg_test"]

depths = data["depths"]
Y_mean, Y_std = data["Y_mean"], data["Y_std"]

print("Train:", X_train.shape, Y_train.shape)
print("Val:  ", X_val.shape, Y_val.shape)
print("Test: ", X_test.shape, Y_test.shape)

# ---------------- 2. NORMALIZE INPUTS ----------------
# Calculate statistics ONLY from training data

X_mean = X_train.mean(axis=(0, 2, 3), keepdims=True)
X_std  = X_train.std(axis=(0, 2, 3), keepdims=True)
X_std[X_std < 1e-6] = 1

X_train = (X_train - X_mean) / X_std
X_val   = (X_val   - X_mean) / X_std
X_test  = (X_test  - X_mean) / X_std

# ---------------- 3. DATALOADERS ----------------

def tensor_loader(X, Y, batch_size=4, shuffle=False):
    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)
    return DataLoader(TensorDataset(X, Y),
                      batch_size=batch_size,
                      shuffle=shuffle)

train_loader = tensor_loader(X_train, Y_train, 4, True)
val_loader   = tensor_loader(X_val, Y_val, 4)
test_loader  = tensor_loader(X_test, Y_test, 4)

# ---------------- 4. DEVICE ----------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

# ============================================================
# 5. ENCODER
# ============================================================

class Encoder(nn.Module):

    def __init__(self, in_channels=11):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.block2 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.block3 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

    def forward(self, x):

        s1 = self.block1(x)      # 32 × 100 × 240
        s2 = self.block2(s1)     # 64 × 50 × 120
        z  = self.block3(s2)     # 128 × 25 × 60

        return z, s1, s2


# ============================================================
# 6. DECODER
# ============================================================

class Decoder(nn.Module):

    def __init__(self, out_channels=15):
        super().__init__()

        self.up1 = nn.ConvTranspose2d(
            128, 64, kernel_size=2, stride=2
        )

        self.conv1 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.up2 = nn.ConvTranspose2d(
            64, 32, kernel_size=2, stride=2
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.output = nn.Conv2d(
            32, out_channels, kernel_size=1
        )

    def forward(self, z, s1, s2):

        # 25×60 → 50×120
        x = self.up1(z)

        # Combine decoder information with encoder features
        x = torch.cat([x, s2], dim=1)
        x = self.conv1(x)

        # 50×120 → 100×240
        x = self.up2(x)

        x = torch.cat([x, s1], dim=1)
        x = self.conv2(x)

        return self.output(x)


# ============================================================
# 7. COMPLETE ENCODER-DECODER MODEL
# ============================================================

class OceanEmbed(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = Encoder(in_channels=11)
        self.decoder = Decoder(out_channels=15)

    def forward(self, x):

        # INPUT:
        # (Batch, 11, 100, 240)

        z, s1, s2 = self.encoder(x)

        # z = compressed learned representation
        # (Batch, 128, 25, 60)

        output = self.decoder(z, s1, s2)

        # OUTPUT:
        # (Batch, 15, 100, 240)

        return output


# ---------------- 8. INITIALIZE ----------------

model = OceanEmbed().to(device)

print("\nModel created!")

sample = torch.randn(2, 11, 100, 240).to(device)

with torch.no_grad():
    prediction = model(sample)

print("Input shape: ", sample.shape)
print("Output shape:", prediction.shape)

# ============================================================
# 9. LOSS + OPTIMIZER
# ============================================================

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ============================================================
# 10. EARLY STOPPING
# ============================================================
# Tracks validation loss across epochs. Saves the model's weights
# to disk every time val_loss improves by more than `min_delta`.
# If it goes `patience` epochs in a row without a meaningful
# improvement, sets `early_stop = True` so the training loop knows
# to break.

class EarlyStopping:

    def __init__(self, patience=10, min_delta=1e-4,
                 path="best_oceanembed_model.pt", verbose=True):

        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.verbose = verbose

        self.best_loss = float("inf")
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss, model):

        improved = val_loss < (self.best_loss - self.min_delta)

        if improved:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.path)

            if self.verbose:
                print(f"  ↳ val_loss improved to {val_loss:.5f} — model saved")

        else:
            self.counter += 1

            if self.verbose:
                print(f"  ↳ no improvement ({self.counter}/{self.patience})")

            if self.counter >= self.patience:
                self.early_stop = True


# ============================================================
# 11. TRAINING + BACKPROPAGATION
# ============================================================

EPOCHS = 2000
PATIENCE = 10
MIN_DELTA = 1e-4

early_stopping = EarlyStopping(
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    path="best_oceanembed_model.pt"
)

history = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS):

    # ---------------- TRAIN ----------------

    model.train()
    train_loss = 0

    for X, Y in train_loader:

        X = X.to(device)
        Y = Y.to(device)

        # 1. Forward pass
        prediction = model(X)

        # 2. Calculate error
        loss = criterion(prediction, Y)

        # 3. Clear old gradients
        optimizer.zero_grad()

        # 4. Backpropagation
        loss.backward()

        # 5. Update model weights
        optimizer.step()

        train_loss += loss.item() * X.size(0)

    train_loss /= len(train_loader.dataset)

    # ---------------- VALIDATION ----------------

    model.eval()
    val_loss = 0

    with torch.no_grad():

        for X, Y in val_loader:

            X = X.to(device)
            Y = Y.to(device)

            prediction = model(X)

            loss = criterion(prediction, Y)

            val_loss += loss.item() * X.size(0)

    val_loss /= len(val_loader.dataset)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train: {train_loss:.5f} | "
        f"Val: {val_loss:.5f}"
    )

    # ---------------- EARLY STOPPING ----------------

    early_stopping(val_loss, model)

    if early_stopping.early_stop:
        print("\nEarly stopping triggered!")
        break


# ============================================================
# 12. LOAD BEST MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        early_stopping.path,
        map_location=device,
        weights_only=True
    )
)

model.eval()


# ============================================================
# 13. TEST SET PREDICTIONS
# ============================================================

predictions = []
actuals = []

with torch.no_grad():

    for X, Y in test_loader:

        X = X.to(device)

        pred = model(X).cpu().numpy()

        predictions.append(pred)
        actuals.append(Y.numpy())

Y_pred = np.concatenate(predictions)
Y_true = np.concatenate(actuals)


# ============================================================
# 14. CONVERT BACK TO REAL TEMPERATURE SCALE
# ============================================================

Y_pred_real = (
    Y_pred * Y_std[None, :, None, None]
    + Y_mean[None, :, None, None]
)

Y_true_real = (
    Y_true * Y_std[None, :, None, None]
    + Y_mean[None, :, None, None]
)


# ============================================================
# 15. FINAL METRICS
# ============================================================

rmse = np.sqrt(
    np.mean((Y_pred_real - Y_true_real) ** 2)
)

mae = np.mean(
    np.abs(Y_pred_real - Y_true_real)
)

print("\n" + "=" * 50)
print("FINAL TEST RESULTS")
print("=" * 50)

print(f"Overall RMSE: {rmse:.4f} °C")
print(f"Overall MAE : {mae:.4f} °C")


# RMSE FOR EACH DEPTH

print("\nRMSE BY DEPTH")

for i, depth in enumerate(depths):

    depth_rmse = np.sqrt(
        np.mean(
            (Y_pred_real[:, i] - Y_true_real[:, i]) ** 2
        )
    )

    print(
        f"Depth {depth:>6.1f} m → "
        f"RMSE: {depth_rmse:.4f} °C"
    )


print("\nTraining complete!")


Train: (48, 11, 100, 240) (48, 15, 100, 240)
Val:   (6, 11, 100, 240) (6, 15, 100, 240)
Test:  (6, 11, 100, 240) (6, 15, 100, 240)

Device: cuda

Model created!
Input shape:  torch.Size([2, 11, 100, 240])
Output shape: torch.Size([2, 15, 100, 240])
Epoch 01/2000 | Train: 0.40635 | Val: 0.31105
  ↳ val_loss improved to 0.31105 — model saved
Epoch 02/2000 | Train: 0.23074 | Val: 0.19946
  ↳ val_loss improved to 0.19946 — model saved
Epoch 03/2000 | Train: 0.16532 | Val: 0.18350
  ↳ val_loss improved to 0.18350 — model saved
Epoch 04/2000 | Train: 0.13113 | Val: 0.13603
  ↳ val_loss improved to 0.13603 — model saved
Epoch 05/2000 | Train: 0.10514 | Val: 0.12655
  ↳ val_loss improved to 0.12655 — model saved
Epoch 06/2000 | Train: 0.09036 | Val: 0.09915
  ↳ val_loss improved to 0.09915 — model saved
Epoch 07/2000 | Train: 0.08040 | Val: 0.09223
  ↳ val_loss improved to 0.09223 — model saved
Epoch 08/2000 | Train: 0.07421 | Val: 0.08903
  ↳ val_loss improved to 0.08903 — model saved
Epoch 0

In [ ]:
# ============================================================
# BASELINE OCEANEMBED MODEL
# 11 Surface Variables → Encoder → Latent Features → Decoder
# → 15 Depth-wise Temperature Maps
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ---------------- 1. LOAD DATA ----------------

DATA_PATH = "/content/ocean_data/ML_DATA/cnn_20250101_20250301.npz"
data = np.load(DATA_PATH)

X_train, Y_train = data["Xg_train"], data["Yg_train"]
X_val,   Y_val   = data["Xg_val"],   data["Yg_val"]
X_test,  Y_test  = data["Xg_test"],  data["Yg_test"]

depths = data["depths"]
Y_mean, Y_std = data["Y_mean"], data["Y_std"]

print("Train:", X_train.shape, Y_train.shape)
print("Val:  ", X_val.shape, Y_val.shape)
print("Test: ", X_test.shape, Y_test.shape)

# ---------------- 2. NORMALIZE INPUTS ----------------
# Calculate statistics ONLY from training data

X_mean = X_train.mean(axis=(0, 2, 3), keepdims=True)
X_std  = X_train.std(axis=(0, 2, 3), keepdims=True)
X_std[X_std < 1e-6] = 1

X_train = (X_train - X_mean) / X_std
X_val   = (X_val   - X_mean) / X_std
X_test  = (X_test  - X_mean) / X_std

# ---------------- 3. DATALOADERS ----------------

def tensor_loader(X, Y, batch_size=4, shuffle=False):
    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)
    return DataLoader(TensorDataset(X, Y),
                      batch_size=batch_size,
                      shuffle=shuffle)

train_loader = tensor_loader(X_train, Y_train, 4, True)
val_loader   = tensor_loader(X_val, Y_val, 4)
test_loader  = tensor_loader(X_test, Y_test, 4)

# ---------------- 4. DEVICE ----------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

# ============================================================
# 5. U-NET BUILDING BLOCKS
# ============================================================

class DoubleConv(nn.Module):
    """(Conv2d → BatchNorm → ReLU) × 2 — the standard U-Net block."""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()

        if mid_channels is None:
            mid_channels = out_channels

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(mid_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    """Downscale: maxpool(2) → DoubleConv."""

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.block(x)


class Up(nn.Module):
    """Upscale via transposed conv, concat with the matching skip
    connection, then DoubleConv. Pads the upsampled tensor if its
    spatial size lands a pixel short of the skip — happens whenever
    an odd dimension gets halved by maxpool (e.g. 25 → 12 → 24)."""

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels, in_channels // 2, kernel_size=2, stride=2
        )
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):

        x = self.up(x)

        diffY = skip.size(2) - x.size(2)
        diffX = skip.size(3) - x.size(3)

        x = F.pad(x, [diffX // 2, diffX - diffX // 2,
                       diffY // 2, diffY - diffY // 2])

        x = torch.cat([skip, x], dim=1)

        return self.conv(x)


# ============================================================
# 6. U-NET
# ============================================================

class UNet(nn.Module):

    def __init__(self, in_channels=11, out_channels=15, base_channels=32):
        super().__init__()

        c1 = base_channels        # 32
        c2 = base_channels * 2     # 64
        c3 = base_channels * 4     # 128
        c4 = base_channels * 8     # 256  (bottleneck)

        self.inc   = DoubleConv(in_channels, c1)   # c1 × 100 × 240
        self.down1 = Down(c1, c2)                   # c2 ×  50 × 120
        self.down2 = Down(c2, c3)                     # c3 ×  25 ×  60
        self.down3 = Down(c3, c4)                      # c4 ×  12 ×  30  (bottleneck)

        self.up3 = Up(c4, c3)                           # → c3 × 25 × 60
        self.up2 = Up(c3, c2)                             # → c2 × 50 × 120
        self.up1 = Up(c2, c1)                               # → c1 × 100 × 240

        self.outc = nn.Conv2d(c1, out_channels, kernel_size=1)

    def forward(self, x):

        s1 = self.inc(x)
        s2 = self.down1(s1)
        s3 = self.down2(s2)
        b  = self.down3(s3)

        x = self.up3(b, s3)
        x = self.up2(x, s2)
        x = self.up1(x, s1)

        return self.outc(x)


# ============================================================
# 7. COMPLETE MODEL
# ============================================================

class OceanEmbed(nn.Module):
    """Thin wrapper so the rest of the script (model = OceanEmbed(),
    training loop, checkpointing) doesn't need to change."""

    def __init__(self):
        super().__init__()

        self.unet = UNet(in_channels=11, out_channels=15, base_channels=32)

    def forward(self, x):

        # INPUT:  (Batch, 11, 100, 240)
        # OUTPUT: (Batch, 15, 100, 240)

        return self.unet(x)


# ---------------- 8. INITIALIZE ----------------

model = OceanEmbed().to(device)

print("\nModel created!")

sample = torch.randn(2, 11, 100, 240).to(device)

with torch.no_grad():
    prediction = model(sample)

print("Input shape: ", sample.shape)
print("Output shape:", prediction.shape)

# ============================================================
# 9. LOSS + OPTIMIZER
# ============================================================

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ============================================================
# 10. EARLY STOPPING
# ============================================================
# Tracks validation loss across epochs. Saves the model's weights
# to disk every time val_loss improves by more than `min_delta`.
# If it goes `patience` epochs in a row without a meaningful
# improvement, sets `early_stop = True` so the training loop knows
# to break.

class EarlyStopping:

    def __init__(self, patience=10, min_delta=1e-4,
                 path="best_oceanembed_model.pt", verbose=True):

        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.verbose = verbose

        self.best_loss = float("inf")
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss, model):

        improved = val_loss < (self.best_loss - self.min_delta)

        if improved:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.path)

            if self.verbose:
                print(f"  ↳ val_loss improved to {val_loss:.5f} — model saved")

        else:
            self.counter += 1

            if self.verbose:
                print(f"  ↳ no improvement ({self.counter}/{self.patience})")

            if self.counter >= self.patience:
                self.early_stop = True


# ============================================================
# 11. TRAINING + BACKPROPAGATION
# ============================================================

EPOCHS = 2000
PATIENCE = 10
MIN_DELTA = 1e-4

early_stopping = EarlyStopping(
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    path="best_oceanembed_model.pt"
)

history = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS):

    # ---------------- TRAIN ----------------

    model.train()
    train_loss = 0

    for X, Y in train_loader:

        X = X.to(device)
        Y = Y.to(device)

        # 1. Forward pass
        prediction = model(X)

        # 2. Calculate error
        loss = criterion(prediction, Y)

        # 3. Clear old gradients
        optimizer.zero_grad()

        # 4. Backpropagation
        loss.backward()

        # 5. Update model weights
        optimizer.step()

        train_loss += loss.item() * X.size(0)

    train_loss /= len(train_loader.dataset)

    # ---------------- VALIDATION ----------------

    model.eval()
    val_loss = 0

    with torch.no_grad():

        for X, Y in val_loader:

            X = X.to(device)
            Y = Y.to(device)

            prediction = model(X)

            loss = criterion(prediction, Y)

            val_loss += loss.item() * X.size(0)

    val_loss /= len(val_loader.dataset)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train: {train_loss:.5f} | "
        f"Val: {val_loss:.5f}"
    )

    # ---------------- EARLY STOPPING ----------------

    early_stopping(val_loss, model)

    if early_stopping.early_stop:
        print("\nEarly stopping triggered!")
        break


# ============================================================
# 12. LOAD BEST MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        early_stopping.path,
        map_location=device,
        weights_only=True
    )
)

model.eval()


# ============================================================
# 13. TEST SET PREDICTIONS
# ============================================================

predictions = []
actuals = []

with torch.no_grad():

    for X, Y in test_loader:

        X = X.to(device)

        pred = model(X).cpu().numpy()

        predictions.append(pred)
        actuals.append(Y.numpy())

Y_pred = np.concatenate(predictions)
Y_true = np.concatenate(actuals)


# ============================================================
# 14. CONVERT BACK TO REAL TEMPERATURE SCALE
# ============================================================

Y_pred_real = (
    Y_pred * Y_std[None, :, None, None]
    + Y_mean[None, :, None, None]
)

Y_true_real = (
    Y_true * Y_std[None, :, None, None]
    + Y_mean[None, :, None, None]
)


# ============================================================
# 15. FINAL METRICS
# ============================================================

rmse = np.sqrt(
    np.mean((Y_pred_real - Y_true_real) ** 2)
)

mae = np.mean(
    np.abs(Y_pred_real - Y_true_real)
)

print("\n" + "=" * 50)
print("FINAL TEST RESULTS")
print("=" * 50)

print(f"Overall RMSE: {rmse:.4f} °C")
print(f"Overall MAE : {mae:.4f} °C")


# RMSE FOR EACH DEPTH

print("\nRMSE BY DEPTH")

for i, depth in enumerate(depths):

    depth_rmse = np.sqrt(
        np.mean(
            (Y_pred_real[:, i] - Y_true_real[:, i]) ** 2
        )
    )

    print(
        f"Depth {depth:>6.1f} m → "
        f"RMSE: {depth_rmse:.4f} °C"
    )


print("\nTraining complete!")

Train: (48, 11, 100, 240) (48, 15, 100, 240)
Val:   (6, 11, 100, 240) (6, 15, 100, 240)
Test:  (6, 11, 100, 240) (6, 15, 100, 240)

Device: cuda

Model created!
Input shape:  torch.Size([2, 11, 100, 240])
Output shape: torch.Size([2, 15, 100, 240])
Epoch 01/2000 | Train: 0.37542 | Val: 0.29121
  ↳ val_loss improved to 0.29121 — model saved
Epoch 02/2000 | Train: 0.20619 | Val: 0.31762
  ↳ no improvement (1/10)
Epoch 03/2000 | Train: 0.14330 | Val: 0.17352
  ↳ val_loss improved to 0.17352 — model saved
Epoch 04/2000 | Train: 0.11106 | Val: 0.12992
  ↳ val_loss improved to 0.12992 — model saved
Epoch 05/2000 | Train: 0.09045 | Val: 0.10604
  ↳ val_loss improved to 0.10604 — model saved
Epoch 06/2000 | Train: 0.07699 | Val: 0.08327
  ↳ val_loss improved to 0.08327 — model saved
Epoch 07/2000 | Train: 0.06826 | Val: 0.07931
  ↳ val_loss improved to 0.07931 — model saved
Epoch 08/2000 | Train: 0.06310 | Val: 0.07086
  ↳ val_loss improved to 0.07086 — model saved
Epoch 09/2000 | Train: 0.059

In [ ]:
# -- Self-contained ML helpers (no external package needed) --
# Everything lives under OUTPUT_ROOT (set by the Setup cell).
from pathlib import Path as _Path


def resolve_root(default="ocean_data"):
    return OUTPUT_ROOT


def ml_dir(root=None):
    d = (_Path(root) if root else resolve_root()) / "ML_DATA"
    d.mkdir(parents=True, exist_ok=True)
    return d


def models_dir(root=None):
    d = (_Path(root) if root else resolve_root()) / "ML_MODELS"
    d.mkdir(parents=True, exist_ok=True)
    return d


def load_splits(path_or_root=None):
    """Load cached splits (path to .npz, data root, or default root)."""
    import numpy as _np
    if isinstance(path_or_root, (str, _Path)) and str(path_or_root).endswith(".npz"):
        path = _Path(path_or_root)
    else:
        root = _Path(path_or_root) if path_or_root else resolve_root()
        cands = sorted((root / "ML_DATA").glob("splits_*.npz"))
        if not cands:
            raise FileNotFoundError(f"No splits found under {root / 'ML_DATA'}")
        path = cands[-1]
    z = _np.load(path, allow_pickle=False)
    return {k: z[k] for k in z.files}

"""PyTorch image-to-image models: 11 surface channels → 15 depth levels.

Architectures: unet (4-scale + skips), resnet (full-resolution residual),
attention_unet (gated skips), densenet (dense reuse, full resolution),
simple_cnn (multi-scale conv baseline). All preserve H×W.
"""

import logging
import time

log = logging.getLogger(__name__)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    torch = None

if HAS_TORCH:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
else:
    DEVICE = "cpu"


if HAS_TORCH:
    class _Block(nn.Module):
        def __init__(self, c_in, c_out):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
                nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
                nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
                nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

        def forward(self, x):
            return self.net(x)

    def _match(src, like):
        if src.shape[2:] == like.shape[2:]:
            return src
        return F.interpolate(src, size=like.shape[2:], mode="bilinear",
                             align_corners=False)

    class UNet(nn.Module):
        """4-scale encoder-decoder with skip connections (size-matched)."""

        def __init__(self, n_features=11, n_depths=15, base_filters=32):
            super().__init__()
            b = base_filters
            self.e1 = _Block(n_features, b)
            self.e2 = _Block(b, b * 2)
            self.e3 = _Block(b * 2, b * 4)
            self.mid = _Block(b * 4, b * 8)
            self.d3 = _Block(b * 8 + b * 4, b * 4)
            self.d2 = _Block(b * 4 + b * 2, b * 2)
            self.d1 = _Block(b * 2 + b, b)
            self.head = nn.Conv2d(b, n_depths, 1)
            self.pool = nn.MaxPool2d(2)

        def forward(self, x):
            s1 = self.e1(x)
            s2 = self.e2(self.pool(s1))
            s3 = self.e3(self.pool(s2))
            m = self.mid(self.pool(s3))
            u3 = self.d3(torch.cat([_match(m, s3), s3], 1))
            u2 = self.d2(torch.cat([_match(u3, s2), s2], 1))
            u1 = self.d1(torch.cat([_match(u2, s1), s1], 1))
            return self.head(_match(u1, x))

    class _AttGate(nn.Module):
        def __init__(self, c_skip, c_gate, c_mid):
            super().__init__()
            self.w_s = nn.Conv2d(c_skip, c_mid, 1, bias=False)
            self.w_g = nn.Conv2d(c_gate, c_mid, 1, bias=False)
            self.psi = nn.Sequential(nn.Conv2d(c_mid, 1, 1), nn.Sigmoid())

        def forward(self, skip, gate):
            g = _match(gate, skip)
            a = self.psi(F.relu(self.w_s(skip) + self.w_g(g)))
            return skip * a

    class AttentionUNet(nn.Module):
        """U-Net with learned attention gates on skips."""

        def __init__(self, n_features=11, n_depths=15, base_filters=32):
            super().__init__()
            b = base_filters
            self.e1 = _Block(n_features, b)
            self.e2 = _Block(b, b * 2)
            self.e3 = _Block(b * 2, b * 4)
            self.mid = _Block(b * 4, b * 8)
            self.g3 = _AttGate(b * 4, b * 8, b * 4)
            self.g2 = _AttGate(b * 2, b * 4, b * 2)
            self.g1 = _AttGate(b, b * 2, b)
            self.d3 = _Block(b * 8 + b * 4, b * 4)
            self.d2 = _Block(b * 4 + b * 2, b * 2)
            self.d1 = _Block(b * 2 + b, b)
            self.head = nn.Conv2d(b, n_depths, 1)
            self.pool = nn.MaxPool2d(2)

        def forward(self, x):
            s1 = self.e1(x)
            s2 = self.e2(self.pool(s1))
            s3 = self.e3(self.pool(s2))
            m = self.mid(self.pool(s3))
            u3 = self.d3(torch.cat([self.g3(s3, m), _match(m, s3)], 1))
            u2 = self.d2(torch.cat([self.g2(s2, u3), _match(u3, s2)], 1))
            u1 = self.d1(torch.cat([self.g1(s1, u2), _match(u2, s1)], 1))
            return self.head(_match(u1, x))

    class _ResBlock(nn.Module):
        def __init__(self, c):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(c, c, 3, padding=1, bias=False),
                nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                nn.Conv2d(c, c, 3, padding=1, bias=False),
                nn.BatchNorm2d(c))

        def forward(self, x):
            return F.relu(x + self.net(x))

    class ResNet(nn.Module):
        """Fully-convolutional residual net, no pooling (full resolution)."""

        def __init__(self, n_features=11, n_depths=15, base_filters=32,
                     n_blocks=8):
            super().__init__()
            b = base_filters
            self.stem = _Block(n_features, b)
            self.body = nn.Sequential(*[_ResBlock(b) for _ in range(n_blocks)])
            self.head = nn.Conv2d(b, n_depths, 1)

        def forward(self, x):
            return self.head(self.body(self.stem(x)))

    class _DenseLayer(nn.Module):
        def __init__(self, c_in, growth_rate=16):
            super().__init__()
            self.net = nn.Sequential(
                nn.BatchNorm2d(c_in), nn.ReLU(inplace=True),
                nn.Conv2d(c_in, growth_rate, 3, padding=1, bias=False))

        def forward(self, x):
            return torch.cat([x, self.net(x)], 1)

    class DenseNet(nn.Module):
        """Dense feature reuse (all-to-all within blocks), full resolution."""

        def __init__(self, n_features=11, n_depths=15, base_filters=32,
                     n_dense_blocks=3, growth_rate=16, layers_per_block=4):
            super().__init__()
            c = base_filters
            self.stem = _Block(n_features, c)
            blocks, trans = [], []
            for _ in range(n_dense_blocks):
                blk = []
                for _ in range(layers_per_block):
                    blk.append(_DenseLayer(c, growth_rate))
                    c += growth_rate
                blocks.append(nn.Sequential(*blk))
                trans.append(nn.Conv2d(c, base_filters, 1, bias=False))
                c = base_filters
            self.blocks = nn.ModuleList(blocks)
            self.trans = nn.ModuleList(trans)
            self.head = nn.Conv2d(c, n_depths, 1)

        def forward(self, x):
            h = self.stem(x)
            for blk, tr in zip(self.blocks, self.trans):
                h = tr(blk(h)) + h  # transition back + residual
            return self.head(h)

    class SimpleCNN(nn.Module):
        """Multi-scale conv baseline (fastest training)."""

        def __init__(self, n_features=11, n_depths=15, base_filters=32):
            super().__init__()
            b = base_filters
            self.net = nn.Sequential(
                _Block(n_features, b),
                _Block(b, b * 2),
                _Block(b * 2, b * 2),
                nn.Conv2d(b * 2, n_depths, 1))

        def forward(self, x):
            return self.net(x)

    _MODELS = {"unet": UNet, "attention_unet": AttentionUNet, "resnet": ResNet,
               "densenet": DenseNet, "simple_cnn": SimpleCNN}


def get_model(name, **kwargs):
    if not HAS_TORCH:
        raise ImportError("torch is not installed")
    try:
        return _MODELS[name](**kwargs).to(DEVICE)
    except KeyError:
        raise ValueError(f"Unknown model '{name}'. Choose from {sorted(_MODELS)}")


def create_dataloaders(X_train, Y_train, X_val, Y_val, batch_size=16,
                       num_workers=0):
    if not HAS_TORCH:
        raise ImportError("torch is not installed")
    to_t = lambda a: torch.from_numpy(__import__("numpy").ascontiguousarray(a)).float()
    train = DataLoader(TensorDataset(to_t(X_train), to_t(Y_train)),
                       batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val = DataLoader(TensorDataset(to_t(X_val), to_t(Y_val)),
                     batch_size=batch_size, num_workers=num_workers)
    return train, val


def _depth_weights(depths, scale=500.0):
    import numpy as np
    w = np.exp(-np.asarray(depths, dtype=np.float64) / scale)
    return (w / w.mean()).astype(np.float32)


def train_model(model, train_loader, val_loader, n_depths=15, depths=None,
                epochs=50, lr=1e-3, patience=10):
    """Train with depth-weighted MSE + early stopping. Returns history."""
    import numpy as np
    model.to(DEVICE)
    w = torch.from_numpy(_depth_weights(
        depths if depths is not None else np.arange(n_depths))).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train_loss": [], "val_loss": []}
    best, best_state, bad = float("inf"), None, 0
    for ep in range(epochs):
        model.train()
        tl = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = (((model(xb) - yb) ** 2) * w[None, :, None, None]).mean()
            loss.backward()
            opt.step()
            tl += float(loss.detach()) * len(xb)
        model.eval()
        vl, n = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vl += float((((model(xb) - yb) ** 2)
                             * w[None, :, None, None]).mean()) * len(xb)
                n += len(xb)
        tl /= len(train_loader.dataset)
        vl /= max(n, 1)
        hist["train_loss"].append(tl)
        hist["val_loss"].append(vl)
        log.info("epoch %d/%d train=%.5f val=%.5f", ep + 1, epochs, tl, vl)
        if vl < best:
            best, bad = vl, 0
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                log.info("Early stopping at epoch %d", ep + 1)
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return hist


def predict(model, X, batch_size=32):
    """Batched inference → numpy array on CPU."""
    import numpy as np
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(np.ascontiguousarray(X[i:i + batch_size])).float().to(DEVICE)
            outs.append(model(xb).cpu().numpy())
    return np.concatenate(outs).astype(np.float32)


def save_model(model, name, root=None):
    path = models_dir(root) / f"{name}.pt"
    torch.save(model.state_dict(), path)
    log.info("Saved %s", path)
    return path
# -- 6d. Train all deep learning models ----------------------------------
import logging, time
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

logging.basicConfig(level=logging.INFO, format='%(message)s')

data = load_splits(DATA_PATH)
# Image-to-image models train on the gridded splits (N, C, H, W).
X_train, Y_train = data['Xg_train'], data['Yg_train']
X_val,   Y_val   = data['Xg_val'],   data['Yg_val']
X_test,  Y_test  = data['Xg_test'],  data['Yg_test']
depths = data['depths']
N_DEPTHS = len(depths)

print(f'Device : {DEVICE}')
print(f'Train  : {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

# -- Models to train (comment any out to skip) ----------------------------
MODELS_CFG = {
    'unet':           dict(n_features=11, n_depths=N_DEPTHS, base_filters=32),
    'resnet':         dict(n_features=11, n_depths=N_DEPTHS, base_filters=32, n_blocks=8),
    'attention_unet': dict(n_features=11, n_depths=N_DEPTHS, base_filters=32),
    'densenet':       dict(n_features=11, n_depths=N_DEPTHS, base_filters=32,
                          n_dense_blocks=3, growth_rate=16, layers_per_block=4),
    'simple_cnn':     dict(n_features=11, n_depths=N_DEPTHS, base_filters=32),
}
TRAIN_EPOCHS = 50
PATIENCE     = 10
BATCH_SIZE   = 16
LR           = 1e-3

train_loader, val_loader = create_dataloaders(
    X_train, Y_train, X_val, Y_val, batch_size=BATCH_SIZE)

if 'all_results' not in dir():
    all_results = {}

for model_name, kwargs in MODELS_CFG.items():
    print(f'\n' + '='*58)
    print(f'  Training: {model_name.upper().replace("_"," ")}')
    print('='*58)
    model   = get_model(model_name, **kwargs)
    n_param = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {n_param:,}')
    t0   = time.time()
    hist = train_model(model, train_loader, val_loader,
                           n_depths=N_DEPTHS, depths=depths,
                           epochs=TRAIN_EPOCHS, lr=LR, patience=PATIENCE)
    elapsed = time.time() - t0
    save_model(model, model_name)
    Y_pred_n = predict(model, X_test)
    Y_pred   = Y_pred_n * data['Y_std'] + data['Y_mean']
    Y_true   = Y_test   * data['Y_std'] + data['Y_mean']
    n_d = Y_true.shape[1]
    rmse_d=[np.sqrt(mean_squared_error(Y_true[:,i],Y_pred[:,i])) for i in range(n_d)]
    mae_d =[mean_absolute_error(Y_true[:,i],Y_pred[:,i])         for i in range(n_d)]
    r2_d  =[r2_score(Y_true[:,i],Y_pred[:,i])                    for i in range(n_d)]
    rmse  = np.sqrt(mean_squared_error(Y_true.flatten(), Y_pred.flatten()))
    mae   = mean_absolute_error(Y_true.flatten(), Y_pred.flatten())
    r2    = r2_score(Y_true.flatten(), Y_pred.flatten())
    all_results[model_name] = dict(
        overall_rmse=rmse, overall_mae=mae, overall_r2=r2,
        rmse_by_depth=rmse_d, mae_by_depth=mae_d, r2_by_depth=r2_d,
        history=hist, train_time=elapsed, n_params=n_param,
        Y_pred=Y_pred, Y_true=Y_true)
    print(f'  RMSE={rmse:.4f}C  MAE={mae:.4f}C  R2={r2:.4f}  Time={elapsed:.0f}s')

print('\nAll models trained and saved to ML_MODELS/')

Device : cuda
Train  : (48, 11, 100, 240)  Val: (6, 11, 100, 240)  Test: (6, 11, 100, 240)

  Training: UNET
  Parameters: 1,951,055


ValueError: operands could not be broadcast together with shapes (6,15,100,240) (15,) 

### 6e. Comparative Analysis and Visualisation

In [ ]:
# -- 6e. Comparative analysis: all models --------------------------------
# Run after cells 6c (sklearn) and 6d (deep learning) have both completed.
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PLOT_DIR = Path('./ML_PLOTS')
PLOT_DIR.mkdir(exist_ok=True)
depths = data['depths']

COLORS = {
    'unet':'#2196F3', 'resnet':'#F44336', 'attention_unet':'#4CAF50',
    'densenet':'#FF9800', 'simple_cnn':'#9C27B0',
    'random_forest':'#795548', 'xgboost':'#607D8B',
}
LABELS = {
    'unet':'U-Net', 'resnet':'ResNet', 'attention_unet':'Attention U-Net',
    'densenet':'DenseNet', 'simple_cnn':'Simple CNN',
    'random_forest':'Random Forest', 'xgboost':'XGBoost',
}

# -- 1. Summary table ------------------------------------------------------
print('='*72)
print(f'{"Model":<22} {"Params":>12} {"RMSE(C)":>9} {"MAE(C)":>9} {"R2":>7} {"Time(s)":>8}')
print('-'*72)
for name, res in all_results.items():
    p = f'{res["n_params"]:,}' if res.get('n_params') else '  N/A'
    t = f'{res["train_time"]:.0f}' if res.get('train_time') else '  N/A'
    print(f'{LABELS.get(name,name):<22} {p:>12} '
          f'{res["overall_rmse"]:>9.4f} {res["overall_mae"]:>9.4f} '
          f'{res["overall_r2"]:>7.4f} {t:>8}')
print('='*72)

# -- 2. Metrics by depth --------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=True)
for name, res in all_results.items():
    c, lbl = COLORS.get(name,'grey'), LABELS.get(name,name)
    axes[0].plot(res['rmse_by_depth'], depths, '-', lw=1.8, color=c, label=lbl)
    axes[1].plot(res['mae_by_depth'],  depths, '-', lw=1.8, color=c, label=lbl)
    axes[2].plot(res['r2_by_depth'],   depths, '-', lw=1.8, color=c, label=lbl)
for ax, xlabel, title in zip(axes,
        ['RMSE (C)','MAE (C)','R2'],
        ['RMSE by Depth (lower=better)','MAE by Depth (lower=better)','R2 by Depth (higher=better)']):
    ax.set_xlabel(xlabel, fontsize=11); ax.set_ylabel('Depth (m)', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.invert_yaxis(); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
axes[2].set_xlim(-0.05, 1.05)
fig.suptitle('Model Comparison - Ocean Temperature Profiles\n5N-30N, 45E-105E | GLORYS12v1 target', fontsize=13)
plt.tight_layout()
fig.savefig(PLOT_DIR / 'comparison_by_depth.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: comparison_by_depth.png')

# -- 2b. Mean actual vs. predicted temperature by depth (whole test set) --
first_res    = next(iter(all_results.values()))
mean_actual  = first_res['Y_true'].mean(axis=0)   # (n_depths,) -- same target set for every model
std_actual   = first_res['Y_true'].std(axis=0)

fig, ax = plt.subplots(figsize=(7, 8))
ax.plot(mean_actual, depths, 'k-o', ms=4, lw=2.2, label='Actual (GLORYS) mean', zorder=10)
ax.fill_betweenx(depths, mean_actual - std_actual, mean_actual + std_actual,
                  color='k', alpha=0.08, zorder=1, label='Actual +-1 std')
for name, res in all_results.items():
    mean_pred = res['Y_pred'].mean(axis=0)
    c, lbl = COLORS.get(name, 'grey'), LABELS.get(name, name)
    ax.plot(mean_pred, depths, '--s', ms=3, lw=1.6, color=c, label=lbl)
ax.set_xlabel('Temperature (C)', fontsize=11)
ax.set_ylabel('Depth (m)', fontsize=11)
ax.invert_yaxis()
ax.set_title('Mean Predicted vs. Actual Temperature by Depth\n(averaged over full test set)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(PLOT_DIR / 'mean_temp_profile_by_depth.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: mean_temp_profile_by_depth.png')

# -- 2c. Table: actual mean temperature (C) at each depth ------------------
print('='*72)
header = f'{"Depth(m)":>9}  {"Actual(C)":>10}'
for name in all_results:
    header += f'  {LABELS.get(name,name)[:12]:>12}'
print(header)
for i, d in enumerate(depths):
    row = f'{d:>9.0f}  {mean_actual[i]:>10.3f}'
    for name, res in all_results.items():
        row += f'  {res["Y_pred"].mean(axis=0)[i]:>12.3f}'
    print(row)
print('='*72)

# -- 3. Training curves (DL models only) ----------------------------------
dl = {k:v for k,v in all_results.items() if 'history' in v}
if dl:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, res in dl.items():
        c,lbl,h = COLORS.get(name,'grey'),LABELS.get(name,name),res['history']
        axes[0].plot(h['train_loss'], lw=1.8, color=c, label=lbl)
        axes[1].plot(h['val_loss'],   lw=1.8, color=c, linestyle='--', label=lbl)
    for ax, title in zip(axes, ['Training Loss','Validation Loss']):
        ax.set_xlabel('Epoch'); ax.set_ylabel('Depth-weighted MSE')
        ax.set_title(title, fontweight='bold'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    fig.suptitle('Training Curves - Deep Learning Models', fontsize=13)
    plt.tight_layout()
    fig.savefig(PLOT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show(); print('Saved: training_curves.png')

# -- 4. Bar charts: overall summary ----------------------------------------
names     = list(all_results.keys())
clrs      = [COLORS.get(n,'grey') for n in names]
lbls      = [LABELS.get(n,n) for n in names]
rmse_vals = [all_results[n]['overall_rmse'] for n in names]
r2_vals   = [all_results[n]['overall_r2']   for n in names]
time_vals = [all_results[n].get('train_time') or 0 for n in names]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].bar(lbls,rmse_vals,color=clrs,edgecolor='k',linewidth=0.5)
axes[0].set_title('Overall RMSE (C) - lower is better', fontweight='bold'); axes[0].set_ylabel('RMSE (C)')
axes[1].bar(lbls,r2_vals,color=clrs,edgecolor='k',linewidth=0.5)
axes[1].set_title('Overall R2 - higher is better', fontweight='bold'); axes[1].set_ylim(0, 1.05)
axes[2].bar(lbls,time_vals,color=clrs,edgecolor='k',linewidth=0.5)
axes[2].set_title('Training Time (s)', fontweight='bold'); axes[2].set_ylabel('Seconds')
for ax in axes: ax.tick_params(axis='x', rotation=20); ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Model Summary Comparison', fontsize=13)
plt.tight_layout()
fig.savefig(PLOT_DIR / 'model_summary_bars.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: model_summary_bars.png')

# -- 5. Temperature profiles: top-3 models --------------------------------
top3 = sorted(all_results.items(), key=lambda x: x[1]['overall_rmse'])[:3]
rng  = np.random.default_rng(42)
idx  = int(rng.integers(0, top3[0][1]['Y_true'].shape[0]))
fig, axes = plt.subplots(1, 3, figsize=(15, 8), sharey=True)
for ax, (name, res) in zip(axes, top3):
    ax.plot(res['Y_true'][idx], depths, 'k-o',  ms=3, label='Actual (GLORYS)')
    ax.plot(res['Y_pred'][idx], depths, '--s',  ms=3, color=COLORS.get(name,'red'), label='Predicted')
    ax.set_xlabel('Temperature (C)', fontsize=10); ax.invert_yaxis()
    ax.set_title(f'{LABELS.get(name,name)}\nRMSE={res["overall_rmse"]:.3f}C', fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Depth (m)', fontsize=10)
fig.suptitle('Temperature Profile Predictions - Top 3 Models (single grid cell)', fontsize=12)
plt.tight_layout()
fig.savefig(PLOT_DIR / 'top3_profile_predictions.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: top3_profile_predictions.png')
print(f'\nAll plots saved to {PLOT_DIR.resolve()}')